# UCAP Tool - anonimizacja wideo egocentrycznego (SAM3 + UniDepth)

Ten notatnik uruchamia kompletne narzedzie webowe UCAP (Utility Carve-out of Anonymisation Protocol) na GPU Colaba. Caly ciezki kod (SAM3, UniDepth, serwer FastAPI) dziala tutaj, a interfejs otwierasz w przegladarce przez publiczny link trycloudflare.com.

Przeplyw pracy: wgraj wideo -> ustaw parametry -> przetworz -> obejrzyj podglad z zamazaniem i statystyki -> rozwiaz konflikty (5 opcji decyzji) -> wyrenderuj finalne wideo + dziennik decyzji -> pobierz.

## Szybki start (3 kroki)
1. **Srodowisko wykonawcze > Zmien typ srodowiska > GPU L4** (zalecane; na slabszym GPU bedzie wolniej).
2. W panelu **Sekrety** (ikona klucza po lewej) dodaj sekret `HF_TOKEN` z tokenem Hugging Face (zakres read) i wlacz go dla tego notatnika. Repozytorium `facebook/sam3` jest bramkowane - najpierw popros o dostep: https://huggingface.co/facebook/sam3
3. **Srodowisko wykonawcze > Uruchom wszystko** i kliknij duzy link `https://....trycloudflare.com` wypisany na dole notatnika.

Zostaw karte Colaba otwarta na czas pracy - zamkniecie lub bezczynnosc wylacza serwer.

In [ ]:
# Runtime check - confirm a GPU is attached (L4 expected for this tool:
# SAM3 + UniDepth are resident at the same time, plus encode work).
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print("GPU:", name, "| total VRAM (GB):", round(total, 2))
    if "L4" not in name:
        print("UWAGA: oczekiwano GPU L4 - na tym GPU narzedzie moze dzialac "
              "wolniej lub zabraknac VRAM.")
else:
    print("Brak GPU. Wybierz: Srodowisko wykonawcze > Zmien typ srodowiska > GPU (L4).")

In [ ]:
# Installs: SAM3 (ultralytics) + UniDepth v2 + the FastAPI server stack.
# Pillow is NOT upgraded (>=12 breaks Colab imports).
!pip install -q -U ultralytics huggingface_hub
# UniDepth v2 - pip from source; predicts its own intrinsics (no calibration).
!pip install -q "git+https://github.com/lpiccinelli-eth/UniDepth.git"
!pip install -q -U opencv-python-headless imageio imageio-ffmpeg
!pip install -q -U fastapi uvicorn python-multipart
print("instalacja zakonczona")

In [ ]:
# Hugging Face auth - facebook/sam3 is a GATED repo.
#   1. Request access once: https://huggingface.co/facebook/sam3
#   2. Token (read scope): https://huggingface.co/settings/tokens
#   3. Colab: Secrets panel (key icon) -> add secret HF_TOKEN -> enable for notebook.
# NEVER hardcode a token in a cell - it leaks the moment the file is shared.
from huggingface_hub import login
try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
    print("Autoryzacja HF OK - bramkowane wagi SAM3 sa dostepne.")
except Exception as e:
    print("Autoryzacja HF NIE jest skonfigurowana:", repr(e))
    print("Bez niej pobranie sam3.pt nie powiedzie sie (blad 401).")

In [ ]:
# ============================ CONFIG (edit me) =============================
PORT = 8000
AUTH_TOKEN = ""  # Ustaw niepusty token, aby kazde zadanie API wymagalo tokenu.

# Create the frontend/ folder so the %%writefile cell below can write into it.
import os
os.makedirs("frontend", exist_ok=True)
print("config set | PORT:", PORT, "| AUTH_TOKEN:", "wlaczony" if AUTH_TOKEN else "wylaczony")

In [ ]:
%%writefile ucap_server.py
"""UCAP Tool backend - FastAPI server + SAM3/UniDepth processing engine.

Single-file backend implementing SPEC.md sections 2, 3 and 4. Heavy ML
imports (torch, ultralytics, unidepth, huggingface_hub) are lazy, so this
module imports cleanly on machines without them; GPU endpoints then fail at
runtime with a clear error message.

Engine logic adapted from the proven reference in
code/notebooks/_build_pipeline_notebooks.py (HELPERS apply_redaction,
SAM3_LOAD, PROCESS _instances/process_video, DEPTH_A UniDepth + near_mask).
"""

import collections
import gc
import json
import os
import queue
import re
import shutil
import subprocess
import threading
import time
import traceback
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

import numpy as np
import cv2

from fastapi import Body, FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import HTMLResponse, Response, StreamingResponse

# --------------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------------

VERSION = "1.0"
AUTH_TOKEN = os.environ.get("UCAP_AUTH_TOKEN", "")

DATA_DIR = Path(os.environ.get("UCAP_DATA_DIR", "ucap_data"))
UPLOADS_DIR = DATA_DIR / "uploads"
JOBS_DIR = DATA_DIR / "jobs"

DEFAULT_PARAMS = {
    "prompts": ["face", "licence plate"],
    "score_threshold": 0.6,
    "near_meters": 1.1,
    "use_metric": True,
    "near_percent": 40,
    "overlap_threshold": 0.15,
    "preserve_policy": "carve",
    "redaction": "blur",
    "blur_ksize": 41,
    "pixelate_blocks": 16,
    "fill_bgr": [255, 0, 0],
    "max_frames": None,
    "depth_stride": 1,
    "unidepth_res_level": 9,
    "mark_conflicts_in_preview": True,
}

# RGB palette shared (same order + values) with frontend/index.html.
PALETTE = [(244, 67, 54), (33, 150, 243), (76, 175, 80), (255, 193, 7),
           (156, 39, 176), (0, 188, 212), (255, 87, 34), (205, 220, 57)]

UNIDEPTH_ID = "lpiccinelli/unidepth-v2-vitl14"
JPEG_QUALITY = 85
TRACKER_WINDOW = 15
TRACKER_IOU = 0.3
_CHUNK = 512 * 1024

_TRACK_DECISIONS = ("unblur_track", "blur_track")
_FRAME_DECISIONS = ("unblur_frame", "blur_frame")

# English user-facing warning strings (shown in the UI stats panel).
W_FFMPEG_MISSING = ("ffmpeg not available - video saved without H.264 "
                    "re-encoding (may not play in the browser)")
W_FFMPEG_FAILED = ("ffmpeg re-encoding failed - video saved without "
                   "H.264 (may not play in the browser)")

VIDEOS = {}  # video_id -> meta dict
JOBS = {}    # job_id -> job dict
_LOCK = threading.RLock()

# --------------------------------------------------------------------------
# Small utilities
# --------------------------------------------------------------------------


def _now_iso():
    return datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")


def _read_json(path, default=None):
    try:
        with open(path, "r", encoding="utf-8") as fh:
            return json.load(fh)
    except Exception:
        return default


def _write_json(path, obj):
    # Atomic write: dump to a sibling temp file, then os.replace (atomic on
    # both POSIX and Windows) so readers never see a truncated/partial file.
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    tmp = str(path) + ".tmp"
    with open(tmp, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, indent=2, ensure_ascii=False, default=str)
    os.replace(tmp, str(path))


def _safe_filename(name):
    base = os.path.basename(str(name or "")).strip()
    base = re.sub(r"[^A-Za-z0-9._-]+", "_", base).strip("._")
    return base or "video.mp4"


def _ensure_dirs():
    UPLOADS_DIR.mkdir(parents=True, exist_ok=True)
    JOBS_DIR.mkdir(parents=True, exist_ok=True)


# --------------------------------------------------------------------------
# Guarded torch helpers (module must import without torch installed)
# --------------------------------------------------------------------------


def _torch():
    try:
        import torch
        return torch
    except Exception:
        return None


def _reset_vram():
    """GPU hygiene before each job (reference: reset_vram)."""
    gc.collect()
    torch = _torch()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def _peak_vram_gb():
    torch = _torch()
    if torch is not None and torch.cuda.is_available():
        return round(torch.cuda.max_memory_allocated() / 1e9, 3)
    return None


def _gpu_name():
    torch = _torch()
    try:
        if torch is not None and torch.cuda.is_available():
            return torch.cuda.get_device_name(0)
    except Exception:
        pass
    return None


# --------------------------------------------------------------------------
# Models (lazy singletons; reference: SAM3_LOAD + DEPTH_A)
# --------------------------------------------------------------------------

_SAM3_CACHE = {}   # score_threshold -> predictor (construction is expensive)
_SAM3_PT = None
_UNIDEPTH = None


def _get_sam3(score_threshold):
    global _SAM3_PT
    from huggingface_hub import hf_hub_download
    from ultralytics.models.sam import SAM3VideoSemanticPredictor
    if _SAM3_PT is None:
        _SAM3_PT = hf_hub_download(repo_id="facebook/sam3",
                                   filename="sam3.pt", local_dir=".")
    key = round(float(score_threshold), 4)
    if key not in _SAM3_CACHE:
        overrides = dict(task="segment", mode="predict", model="sam3.pt",
                         half=True, save=False, retina_masks=True, verbose=False)
        _SAM3_CACHE[key] = SAM3VideoSemanticPredictor(
            overrides=overrides, score_threshold_detection=key)
    return _SAM3_CACHE[key]


def _get_unidepth(res_level=9):
    global _UNIDEPTH
    import torch
    if _UNIDEPTH is None:
        from unidepth.models import UniDepthV2
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        _UNIDEPTH = UniDepthV2.from_pretrained(UNIDEPTH_ID).to(dev).eval()
    try:
        _UNIDEPTH.resolution_level = int(res_level)
    except Exception:
        pass
    return _UNIDEPTH


def estimate_depth(frame_bgr, res_level=9):
    """Return a metric depth map (H, W) float32 metres (reference DEPTH_A)."""
    import torch
    model = _get_unidepth(res_level)
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    t = torch.from_numpy(rgb).permute(2, 0, 1)  # C,H,W uint8
    with torch.no_grad():
        pred = model.infer(t)
    return pred["depth"].squeeze().cpu().numpy().astype("float32")


def near_mask(depth, params):
    """Bool mask of the near interaction zone to PRESERVE (reference DEPTH_A)."""
    d = np.asarray(depth, dtype="float32")
    if params.get("use_metric", True):
        return (d > 0) & (d < float(params.get("near_meters", 1.1)))
    valid = d[np.isfinite(d) & (d > 0)]
    if valid.size == 0:
        return np.zeros(d.shape, bool)
    return (d > 0) & (d <= np.percentile(valid, float(params.get("near_percent", 40))))


def preload_models():
    """Download sam3.pt + build the default predictor + load UniDepth."""
    print("UCAP: preloading models (SAM3 + UniDepth)...")
    _get_sam3(DEFAULT_PARAMS["score_threshold"])
    _get_unidepth(DEFAULT_PARAMS["unidepth_res_level"])
    print("UCAP: models ready.")


# --------------------------------------------------------------------------
# Engine helpers (reference: HELPERS + PROCESS)
# --------------------------------------------------------------------------


def _instances(r, H, W):
    """Instance bool masks from a SAM3 result (reference PROCESS _instances)."""
    out = []
    m = getattr(r, "masks", None)
    if m is not None and getattr(m, "data", None) is not None and len(m.data) > 0:
        arr = m.data.to("cpu").numpy().astype(bool)  # (N, h, w)
        for k in range(arr.shape[0]):
            im = arr[k]
            if im.shape != (H, W):
                im = cv2.resize(im.astype("uint8"), (W, H),
                                interpolation=cv2.INTER_NEAREST).astype(bool)
            out.append(im)
    return out


def _apply_redaction(frame_bgr, mask, params):
    """Mask-aware redaction (reference HELPERS apply_redaction)."""
    out = frame_bgr.copy()
    if mask is None or not mask.any():
        return out
    mode = params.get("redaction", "blur")
    if mode == "fill":
        out[mask] = np.array(params.get("fill_bgr", [255, 0, 0]), dtype=np.uint8)
    elif mode == "pixelate":
        h, w = frame_bgr.shape[:2]
        s = max(1, int(params.get("pixelate_blocks", 16)))
        small = cv2.resize(frame_bgr, (max(1, w // s), max(1, h // s)),
                           interpolation=cv2.INTER_LINEAR)
        pix = cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)
        out[mask] = pix[mask]
    else:  # "blur" (default)
        k = int(params.get("blur_ksize", 41)) | 1
        out[mask] = cv2.GaussianBlur(frame_bgr, (k, k), 0)[mask]
    return out


class _IoUTracker:
    """Fallback tracker: greedy IoU match of current instance masks against
    instances seen in the last TRACKER_WINDOW frames (one-to-one, IoU desc).
    Masks are downscaled for the IoU computation to keep it cheap."""

    def __init__(self, window=TRACKER_WINDOW, iou_threshold=TRACKER_IOU, max_side=256):
        self.window = window
        self.iou_threshold = iou_threshold
        self.max_side = max_side
        self.history = collections.deque()  # (frame_idx, {tid: small_mask})
        self.next_id = 0

    def _small(self, mask):
        h, w = mask.shape[:2]
        if max(h, w) <= self.max_side:
            return mask.astype(bool)
        s = self.max_side / float(max(h, w))
        return cv2.resize(mask.astype(np.uint8),
                          (max(1, int(w * s)), max(1, int(h * s))),
                          interpolation=cv2.INTER_NEAREST).astype(bool)

    def _prune(self, frame_idx):
        while self.history and self.history[0][0] < frame_idx - self.window:
            self.history.popleft()

    def _latest(self):
        latest = {}
        for _, d in self.history:  # oldest -> newest, newest wins
            latest.update(d)
        return latest

    def assign(self, insts, frame_idx):
        self._prune(frame_idx)
        small = [self._small(m) for m in insts]
        latest = self._latest()
        pairs = []
        for i, sm in enumerate(small):
            a = int(sm.sum())
            for tid, om in latest.items():
                if om.shape != sm.shape:
                    om = cv2.resize(om.astype(np.uint8), (sm.shape[1], sm.shape[0]),
                                    interpolation=cv2.INTER_NEAREST).astype(bool)
                inter = int((sm & om).sum())
                union = a + int(om.sum()) - inter
                iou = (inter / union) if union else 0.0
                if iou >= self.iou_threshold:
                    pairs.append((iou, i, tid))
        pairs.sort(key=lambda p: p[0], reverse=True)
        ids = [None] * len(insts)
        used = set()
        for _, i, tid in pairs:
            if ids[i] is None and tid not in used:
                ids[i] = tid
                used.add(tid)
        for i in range(len(insts)):
            if ids[i] is None:
                ids[i] = self.next_id
                self.next_id += 1
        self.history.append((frame_idx, {tid: small[i] for i, tid in enumerate(ids)}))
        return ids

    def observe(self, insts, ids, frame_idx):
        """Feed externally-provided (SAM3) ids into the history so a momentary
        id dropout can still be bridged by IoU matching."""
        self._prune(frame_idx)
        self.history.append((frame_idx, {tid: self._small(m)
                                         for m, tid in zip(insts, ids)}))
        if ids:
            self.next_id = max(self.next_id, max(ids) + 1)


def _ids_and_labels(r, insts, tracker, prompts, frame_idx):
    """Track ids from r.boxes.id when present, else the IoU fallback tracker;
    labels from r.boxes.cls via r.names (or the prompts list)."""
    n_inst = len(insts)
    boxes = getattr(r, "boxes", None)
    ids = None
    if boxes is not None and getattr(boxes, "id", None) is not None:
        try:
            ids = [int(v) for v in boxes.id.tolist()]
            if len(ids) != n_inst:
                ids = None
        except Exception:
            ids = None
    labels = ["object"] * n_inst
    if boxes is not None and getattr(boxes, "cls", None) is not None:
        try:
            cls = [int(v) for v in boxes.cls.tolist()]
            names = getattr(r, "names", None)
            for i, c in enumerate(cls[:n_inst]):
                if isinstance(names, dict) and c in names:
                    labels[i] = str(names[c])
                elif isinstance(names, (list, tuple)) and 0 <= c < len(names):
                    labels[i] = str(names[c])
                elif 0 <= c < len(prompts):
                    labels[i] = str(prompts[c])
        except Exception:
            pass
    if ids is None:
        ids = tracker.assign(insts, frame_idx)
    else:
        tracker.observe(insts, ids, frame_idx)
    return ids, labels


# --------------------------------------------------------------------------
# ffmpeg re-encode (browser-playable H.264)
# --------------------------------------------------------------------------


def _find_ffmpeg():
    exe = shutil.which("ffmpeg")
    if exe:
        return exe
    try:
        import imageio_ffmpeg
        return imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        return None


def _reencode_h264(raw_path, out_path, warnings):
    """Re-encode raw mp4v to H.264; on failure keep the raw file as out_path."""
    raw_path = Path(raw_path)
    out_path = Path(out_path)
    exe = _find_ffmpeg()
    if exe:
        cmd = [exe, "-y", "-i", str(raw_path), "-c:v", "libx264",
               "-preset", "veryfast", "-crf", "23", "-pix_fmt", "yuv420p",
               "-movflags", "+faststart", str(out_path)]
        try:
            res = subprocess.run(cmd, capture_output=True)
            if res.returncode == 0 and out_path.is_file() and out_path.stat().st_size > 0:
                try:
                    raw_path.unlink()
                except Exception:
                    pass
                return
        except Exception:
            pass
        warnings.append(W_FFMPEG_FAILED)
    else:
        warnings.append(W_FFMPEG_MISSING)
    if raw_path.is_file():
        if out_path.exists():
            out_path.unlink()
        raw_path.rename(out_path)


# --------------------------------------------------------------------------
# Param + decisions validation
# --------------------------------------------------------------------------


def _as_bool(v, default):
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)):
        return bool(v)
    if isinstance(v, str):
        return v.strip().lower() in ("1", "true", "yes", "tak", "on")
    return default


def _as_float(v, default, lo, hi):
    try:
        f = float(v)
    except Exception:
        return default
    if f != f:  # NaN
        return default
    return min(max(f, lo), hi)


def _as_int(v, default, lo, hi):
    try:
        i = int(float(v))
    except Exception:
        return default
    return min(max(i, lo), hi)


def _validate_params(p):
    """Fill defaults + clamp every job param (SPEC 4.1)."""
    if not isinstance(p, dict):
        p = {}
    out = {}
    raw = p.get("prompts", DEFAULT_PARAMS["prompts"])
    if isinstance(raw, str):
        raw = [raw]
    prompts = []
    if isinstance(raw, (list, tuple)):
        prompts = [str(x).strip() for x in raw if str(x).strip()]
    out["prompts"] = prompts or list(DEFAULT_PARAMS["prompts"])
    out["score_threshold"] = _as_float(p.get("score_threshold"), 0.6, 0.01, 0.99)
    out["near_meters"] = _as_float(p.get("near_meters"), 1.1, 0.05, 50.0)
    out["use_metric"] = _as_bool(p.get("use_metric"), True)
    out["near_percent"] = _as_float(p.get("near_percent"), 40, 1, 99)
    out["overlap_threshold"] = _as_float(p.get("overlap_threshold"), 0.15, 0.0, 1.0)
    out["preserve_policy"] = p.get("preserve_policy") if p.get("preserve_policy") in ("carve", "failsafe") else "carve"
    out["redaction"] = p.get("redaction") if p.get("redaction") in ("blur", "pixelate", "fill") else "blur"
    out["blur_ksize"] = _as_int(p.get("blur_ksize"), 41, 3, 301)
    out["pixelate_blocks"] = _as_int(p.get("pixelate_blocks"), 16, 2, 128)
    fb = p.get("fill_bgr", DEFAULT_PARAMS["fill_bgr"])
    if isinstance(fb, (list, tuple)) and len(fb) == 3:
        out["fill_bgr"] = [_as_int(c, 0, 0, 255) for c in fb]
    else:
        out["fill_bgr"] = list(DEFAULT_PARAMS["fill_bgr"])
    mf = p.get("max_frames", None)
    if mf in (None, "", 0, "0", False):
        out["max_frames"] = None
    else:
        try:
            out["max_frames"] = max(1, int(float(mf)))
        except Exception:
            out["max_frames"] = None
    out["depth_stride"] = _as_int(p.get("depth_stride"), 1, 1, 30)
    out["unidepth_res_level"] = _as_int(p.get("unidepth_res_level"), 9, 0, 9)
    out["mark_conflicts_in_preview"] = _as_bool(p.get("mark_conflicts_in_preview"), True)
    return out


def _default_decisions():
    return {"tracks": {}, "frames": {}, "dropped_frames": [],
            "unresolved_policy": "failsafe"}


def _validate_decisions(body):
    """Validate + string-key coerce a decisions payload (SPEC 3). 422 on bad shape."""
    def bad(msg):
        raise HTTPException(status_code=422, detail="invalid decisions: " + msg)
    if not isinstance(body, dict):
        bad("body must be a JSON object")
    out = _default_decisions()
    tracks = body.get("tracks")
    if tracks is None:
        tracks = {}
    if not isinstance(tracks, dict):
        bad("'tracks' must be an object")
    for k, v in tracks.items():
        try:
            tid = int(str(k))
        except Exception:
            bad("track key %r is not an integer" % (k,))
        if v not in _TRACK_DECISIONS:
            bad("tracks[%s] must be one of %s" % (k, list(_TRACK_DECISIONS)))
        out["tracks"][str(tid)] = v
    frames = body.get("frames")
    if frames is None:
        frames = {}
    if not isinstance(frames, dict):
        bad("'frames' must be an object")
    for k, sub in frames.items():
        try:
            n = int(str(k))
        except Exception:
            bad("frame key %r is not an integer" % (k,))
        if not isinstance(sub, dict):
            bad("frames[%s] must be an object" % (k,))
        clean = {}
        for tk, tv in sub.items():
            try:
                tid = int(str(tk))
            except Exception:
                bad("frames[%s] track key %r is not an integer" % (k, tk))
            if tv not in _FRAME_DECISIONS:
                bad("frames[%s][%s] must be one of %s" % (k, tk, list(_FRAME_DECISIONS)))
            clean[str(tid)] = tv
        if clean:
            out["frames"][str(n)] = clean
    dropped = body.get("dropped_frames")
    if dropped is None:
        dropped = []
    if not isinstance(dropped, (list, tuple)):
        bad("'dropped_frames' must be a list")
    seen = set()
    for x in dropped:
        try:
            n = int(x)
        except Exception:
            bad("dropped_frames entry %r is not an integer" % (x,))
        if n >= 0:
            seen.add(n)
    out["dropped_frames"] = sorted(seen)
    pol = body.get("unresolved_policy") or "failsafe"
    if pol not in ("failsafe", "carve"):
        bad("'unresolved_policy' must be 'failsafe' or 'carve'")
    out["unresolved_policy"] = pol
    return out


def _read_decisions(job_dir):
    raw = _read_json(Path(job_dir) / "decisions.json", None)
    if raw is None:
        return _default_decisions()
    try:
        return _validate_decisions(raw)
    except HTTPException:
        return _default_decisions()


# --------------------------------------------------------------------------
# Job registry + state persistence
# --------------------------------------------------------------------------


def _save_state(job):
    _write_json(JOBS_DIR / job["job_id"] / "state.json",
                {"state": job["state"], "error": job["error"],
                 "video_id": job["video_id"], "created_utc": job["created_utc"]})


def _set_state(job, state, error=None):
    job["state"] = state
    job["error"] = error
    _save_state(job)


def _default_progress(frame=0, total=0, stage="pipeline"):
    return {"frame": frame, "total": total, "fps": 0.0, "eta_s": None, "stage": stage}


def _get_job_or_404(job_id):
    job = JOBS.get(job_id)
    if job is None:
        raise HTTPException(status_code=404, detail="unknown job_id")
    return job


def _job_status(job):
    return {"job_id": job["job_id"], "video_id": job["video_id"],
            "video_filename": job["video_filename"], "state": job["state"],
            "created_utc": job["created_utc"], "progress": dict(job["progress"]),
            "params": job["params"], "stats": job["stats"],
            "render_stats": job["render_stats"], "error": job["error"]}


def _video_path(video_id):
    video = VIDEOS.get(video_id)
    if video is None:
        return None
    return UPLOADS_DIR / video_id / video["filename"]


# --------------------------------------------------------------------------
# Processing (SPEC 2.2) - runs on the worker thread
# --------------------------------------------------------------------------


def _process_job(job):
    job_dir = JOBS_DIR / job["job_id"]
    masks_dir = job_dir / "masks"
    masks_dir.mkdir(parents=True, exist_ok=True)
    params = job["params"]
    video = VIDEOS.get(job["video_id"])
    if video is None:
        raise RuntimeError("unknown video " + str(job["video_id"]))
    src = _video_path(job["video_id"])
    fps = float(video.get("fps") or 30.0) or 30.0
    W, H = int(video["width"]), int(video["height"])
    total = int(video.get("frames") or 0)
    if params["max_frames"]:
        total = min(total, params["max_frames"]) if total else params["max_frames"]

    warnings = []
    _set_state(job, "processing")
    job["progress"] = _default_progress(0, total, "pipeline")
    job["_frames_index"] = None

    _reset_vram()
    sam3 = _get_sam3(params["score_threshold"])

    raw_path = job_dir / "preview_raw.mp4"
    writer = cv2.VideoWriter(str(raw_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
    tracker = _IoUTracker()
    frames_meta = []
    tracks = {}
    near = np.zeros((H, W), bool)
    depth_failed_once = False
    depth_stride = max(1, int(params["depth_stride"]))
    n = n_blur = n_flag = n_conf = 0
    t0 = time.perf_counter()

    try:
        for r in sam3(source=str(src), text=list(params["prompts"]), stream=True):
            frame = r.orig_img
            Hf, Wf = frame.shape[:2]
            insts = _instances(r, Hf, Wf)
            tids, labels = _ids_and_labels(r, insts, tracker, params["prompts"], n)

            # Depth + near mask, computed every depth_stride-th frame.
            if n % depth_stride == 0:
                try:
                    depth = estimate_depth(frame, params["unidepth_res_level"])
                    nm = near_mask(depth, params)
                    if nm.shape != (Hf, Wf):
                        nm = cv2.resize(nm.astype("uint8"), (Wf, Hf),
                                        interpolation=cv2.INTER_NEAREST).astype(bool)
                    near = nm
                except Exception as e:
                    near = np.zeros((Hf, Wf), bool)
                    if not depth_failed_once:
                        depth_failed_once = True
                        print("UCAP depth error (continuing with empty near mask):", repr(e))
                        warnings.append("depth unavailable - near zone empty: " + repr(e))
            if near.shape != (Hf, Wf):
                near = np.zeros((Hf, Wf), bool)

            # Merge instances per track id FIRST (SAM3 can in principle return
            # the same id twice on one frame) so each (frame, tid) yields
            # exactly one metadata entry / conflict identity.
            privacy = np.zeros((Hf, Wf), bool)
            obj_masks = {}
            obj_labels = {}
            for im, tid, label in zip(insts, tids, labels):
                if not im.any():
                    continue
                tid = int(tid)
                privacy |= im
                obj_masks[tid] = (obj_masks[tid] | im) if tid in obj_masks else im
                if obj_labels.get(tid, "object") == "object":
                    obj_labels[tid] = label
            inst_meta = []
            frame_conflict = False
            for tid, im in obj_masks.items():
                area = int(im.sum())
                label = obj_labels.get(tid, "object")
                ov = float((im & near).sum()) / area
                conflict = bool(ov >= float(params["overlap_threshold"]))
                x, y, w, h = cv2.boundingRect(im.astype(np.uint8))
                inst_meta.append({"track_id": tid, "label": label,
                                  "area_px": area, "overlap": round(ov, 4),
                                  "conflict": conflict,
                                  "bbox": [int(x), int(y), int(w), int(h)]})
                t = tracks.setdefault(tid, {"track_id": tid, "label": label,
                                            "first_frame": n, "last_frame": n,
                                            "n_frames": 0, "n_conflicts": 0,
                                            "conflict_frames": []})
                t["last_frame"] = n
                t["n_frames"] += 1
                if t["label"] == "object" and label != "object":
                    t["label"] = label
                if conflict:
                    frame_conflict = True
                    n_conf += 1
                    t["n_conflicts"] += 1
                    t["conflict_frames"].append(n)

            # Preview-frame blur mask: the base policy.
            blur = (privacy & ~near) if params["preserve_policy"] == "carve" else privacy.copy()
            out = _apply_redaction(frame, blur, params)
            if frame_conflict and params["mark_conflicts_in_preview"]:
                carved = privacy & near
                if carved.any():
                    cnts, _ = cv2.findContours(carved.astype(np.uint8),
                                               cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    cv2.drawContours(out, cnts, -1, (0, 0, 255), 2)
            if out.shape[0] != H or out.shape[1] != W:
                out = cv2.resize(out, (W, H))
            writer.write(out)
            if blur.any():
                n_blur += 1
            if frame_conflict:
                n_flag += 1

            # Persist per-frame masks for the later re-render.
            np.savez_compressed(masks_dir / ("f%06d.npz" % n), near=near,
                                **{("obj_%d" % t_id): m for t_id, m in obj_masks.items()})
            frames_meta.append({"frame": n, "time_s": round(n / fps, 3),
                                "instances": inst_meta})

            n += 1
            dt = time.perf_counter() - t0
            cur_fps = (n / dt) if dt > 0 else 0.0
            eta = int((total - n) / cur_fps) if (cur_fps > 0 and total > n) else None
            job["progress"] = {"frame": n, "total": total or n, "fps": round(cur_fps, 2),
                               "eta_s": eta, "stage": "pipeline"}
            if params["max_frames"] and n >= params["max_frames"]:
                break
    finally:
        writer.release()

    dt = time.perf_counter() - t0
    job["progress"]["stage"] = "encode"
    _reencode_h264(raw_path, job_dir / "preview.mp4", warnings)

    stats = {"frames": n,
             "fps_pipeline": round(n / dt, 2) if dt > 0 else 0.0,
             "frames_with_blur": n_blur,
             "flagged_frames": n_flag,
             "m2_conflict_rate": round(n_flag / n, 4) if n else 0.0,
             "peak_vram_gb": _peak_vram_gb(),
             "n_tracks": len(tracks),
             "n_conflicts": n_conf,
             "warnings": warnings}
    job["stats"] = stats
    _write_json(job_dir / "stats.json", stats)
    _write_json(job_dir / "frames.json", frames_meta)
    _write_json(job_dir / "tracks.json", {str(k): v for k, v in tracks.items()})
    job["progress"] = {"frame": n, "total": n, "fps": stats["fps_pipeline"],
                       "eta_s": 0, "stage": "pipeline"}
    _set_state(job, "done")


# --------------------------------------------------------------------------
# Final render (SPEC 3.1) - runs on the worker thread
# --------------------------------------------------------------------------


def _collect_conflicts(frames_meta):
    out = []
    for fm in frames_meta:
        for inst in fm.get("instances", []):
            if inst.get("conflict"):
                out.append({"frame": int(fm["frame"]), "time_s": fm.get("time_s", 0.0),
                            "track_id": int(inst["track_id"]),
                            "label": inst.get("label", "object"),
                            "overlap": inst.get("overlap", 0.0),
                            "area_px": inst.get("area_px", 0),
                            "bbox": inst.get("bbox", [0, 0, 0, 0])})
    out.sort(key=lambda c: (c["frame"], c["track_id"]))
    return out


def _render_job(job):
    job_dir = JOBS_DIR / job["job_id"]
    params = job["params"]
    video = VIDEOS.get(job["video_id"])
    if video is None:
        raise RuntimeError("unknown video " + str(job["video_id"]))

    # Read under the lock so a concurrent decisions autosave (POST handler)
    # can never interleave with the read at render start.
    with _LOCK:
        decisions = _read_decisions(job_dir)
    track_dec = {int(k): v for k, v in decisions["tracks"].items()}
    frame_dec = {int(k): {int(t): tv for t, tv in v.items()}
                 for k, v in decisions["frames"].items()}
    dropped = set(int(x) for x in decisions["dropped_frames"])
    policy = decisions.get("unresolved_policy", "failsafe")
    base_policy = params["preserve_policy"]

    frames_meta = _read_json(job_dir / "frames.json", []) or []
    conflicts = _collect_conflicts(frames_meta)
    conflict_set = {(c["frame"], c["track_id"]) for c in conflicts}

    n_proc = (job.get("stats") or {}).get("frames") or len(frames_meta)
    fps = float(video.get("fps") or 30.0) or 30.0
    W, H = int(video["width"]), int(video["height"])
    job["progress"] = _default_progress(0, n_proc, "render")

    src = _video_path(job["video_id"])
    cap = cv2.VideoCapture(str(src))
    raw_path = job_dir / "final_raw.mp4"
    writer = cv2.VideoWriter(str(raw_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
    frames_written = frames_dropped = 0
    t0 = time.perf_counter()
    try:
        for nidx in range(int(n_proc)):
            ok, frame = cap.read()
            if not ok:
                break
            if nidx in dropped:
                frames_dropped += 1
                continue
            blur = np.zeros(frame.shape[:2], bool)
            npz_path = job_dir / "masks" / ("f%06d.npz" % nidx)
            if npz_path.is_file():
                with np.load(str(npz_path)) as npz:
                    near = npz["near"].astype(bool) if "near" in npz.files else np.zeros(frame.shape[:2], bool)
                    if near.shape != frame.shape[:2]:
                        near = np.zeros(frame.shape[:2], bool)
                    for key in npz.files:
                        if not key.startswith("obj_"):
                            continue
                        tid = int(key[4:])
                        mask = npz[key].astype(bool)
                        if mask.shape != frame.shape[:2]:
                            mask = cv2.resize(mask.astype(np.uint8),
                                              (frame.shape[1], frame.shape[0]),
                                              interpolation=cv2.INTER_NEAREST).astype(bool)
                        # Precedence: dropped > frame-local > track >
                        # unresolved-conflict-policy > base policy.
                        eff = frame_dec.get(nidx, {}).get(tid, track_dec.get(tid))
                        if eff in ("unblur_frame", "unblur_track"):
                            continue
                        if eff in ("blur_frame", "blur_track"):
                            blur |= mask
                            continue
                        pol = policy if (nidx, tid) in conflict_set else base_policy
                        blur |= (mask & ~near) if pol == "carve" else mask
            out = _apply_redaction(frame, blur, params)
            if out.shape[0] != H or out.shape[1] != W:
                out = cv2.resize(out, (W, H))
            writer.write(out)
            frames_written += 1
            dt = time.perf_counter() - t0
            cur_fps = ((nidx + 1) / dt) if dt > 0 else 0.0
            eta = int((n_proc - nidx - 1) / cur_fps) if cur_fps > 0 else None
            job["progress"] = {"frame": nidx + 1, "total": n_proc,
                               "fps": round(cur_fps, 2), "eta_s": eta, "stage": "render"}
    finally:
        cap.release()
        writer.release()

    job["progress"]["stage"] = "encode"
    enc_warnings = []
    _reencode_h264(raw_path, job_dir / "final.mp4", enc_warnings)
    if enc_warnings and isinstance(job.get("stats"), dict):
        ws = list(job["stats"].get("warnings") or [])
        for w in enc_warnings:
            if w not in ws:
                ws.append(w)
        job["stats"]["warnings"] = ws
        _write_json(job_dir / "stats.json", job["stats"])

    # Decision log (the M2 audit trail).
    log_conflicts = []
    unresolved = 0
    for c in conflicts:
        nidx, tid = c["frame"], c["track_id"]
        if nidx in dropped:
            resolution, resolved_by = "drop_frame", "drop"
        elif tid in frame_dec.get(nidx, {}):
            resolution, resolved_by = frame_dec[nidx][tid], "frame"
        elif tid in track_dec:
            resolution, resolved_by = track_dec[tid], "track"
        else:
            resolution, resolved_by = policy, "unresolved"
            unresolved += 1
        log_conflicts.append({"frame": nidx, "time_s": c["time_s"],
                              "track_id": tid, "label": c["label"],
                              "overlap": c["overlap"],
                              "resolution": resolution, "resolved_by": resolved_by})
    render_stats = {"frames_written": frames_written,
                    "frames_dropped": frames_dropped,
                    "conflicts_total": len(conflicts),
                    "resolved": len(conflicts) - unresolved,
                    "unresolved": unresolved,
                    "unresolved_policy": policy}
    log = {"tool": "UCAP Tool", "created_utc": _now_iso(),
           "video": {"video_id": job["video_id"], "filename": video["filename"],
                     "fps": fps, "frames": int(video.get("frames") or 0)},
           "params": params, "decisions": decisions,
           "conflicts": log_conflicts, "render_stats": render_stats}
    _write_json(job_dir / "decision_log.json", log)
    job["render_stats"] = render_stats
    _set_state(job, "rendered")


# --------------------------------------------------------------------------
# FIFO worker thread (one job at a time)
# --------------------------------------------------------------------------

_QUEUE = queue.Queue()
_WORKER = None
_WORKER_LOCK = threading.Lock()


def _worker_loop():
    while True:
        kind, job_id = _QUEUE.get()
        job = JOBS.get(job_id)
        try:
            if job is None:
                continue
            if kind == "process":
                _process_job(job)
            elif kind == "render":
                _render_job(job)
        except Exception as e:
            traceback.print_exc()
            if job is not None:
                _set_state(job, "error", error=(str(e) or repr(e)))
        finally:
            _QUEUE.task_done()


def _enqueue(kind, job_id):
    global _WORKER
    with _WORKER_LOCK:
        if _WORKER is None or not _WORKER.is_alive():
            _WORKER = threading.Thread(target=_worker_loop, name="ucap-worker", daemon=True)
            _WORKER.start()
    _QUEUE.put((kind, job_id))


# --------------------------------------------------------------------------
# Disk rescan on server start (SPEC 2.3)
# --------------------------------------------------------------------------


def _rescan_disk():
    _ensure_dirs()
    with _LOCK:
        try:
            vdirs = sorted([d for d in UPLOADS_DIR.iterdir() if d.is_dir()],
                           key=lambda d: d.stat().st_mtime)
        except Exception:
            vdirs = []
        for vdir in vdirs:
            meta = _read_json(vdir / "meta.json", None)
            if not isinstance(meta, dict):
                continue
            vid = str(meta.get("video_id") or vdir.name)
            meta["video_id"] = vid
            if not (vdir / str(meta.get("filename", ""))).is_file():
                continue
            VIDEOS[vid] = meta
        try:
            jdirs = sorted([d for d in JOBS_DIR.iterdir() if d.is_dir()],
                           key=lambda d: d.stat().st_mtime)
        except Exception:
            jdirs = []
        for jdir in jdirs:
            state = _read_json(jdir / "state.json", None)
            if not isinstance(state, dict):
                continue
            st = state.get("state", "error")
            err = state.get("error")
            interrupted = st in ("queued", "processing", "rendering")
            if interrupted:
                st, err = "error", "interrupted by server restart"
            stats = _read_json(jdir / "stats.json", None)
            log = _read_json(jdir / "decision_log.json", None)
            render_stats = log.get("render_stats") if isinstance(log, dict) else None
            vid = str(state.get("video_id", ""))
            created = state.get("created_utc")
            if not created:
                try:
                    created = datetime.fromtimestamp(
                        jdir.stat().st_mtime, tz=timezone.utc
                    ).isoformat(timespec="seconds").replace("+00:00", "Z")
                except Exception:
                    created = _now_iso()
            n_done = (stats or {}).get("frames", 0)
            job = {"job_id": jdir.name, "video_id": vid,
                   "video_filename": (VIDEOS.get(vid) or {}).get("filename", ""),
                   "state": st, "created_utc": created,
                   "params": _validate_params(_read_json(jdir / "params.json", {})),
                   "progress": _default_progress(n_done, n_done, "pipeline"),
                   "stats": stats, "render_stats": render_stats, "error": err}
            JOBS[jdir.name] = job
            if interrupted:
                _save_state(job)


# --------------------------------------------------------------------------
# Hand-rolled HTTP Range support (SPEC 4)
# --------------------------------------------------------------------------


def _parse_range(header, size):
    """Return (start, end) inclusive, 'unsatisfiable', or None (ignore)."""
    if size <= 0:
        return "unsatisfiable"
    try:
        units, _, spec = str(header).partition("=")
        if units.strip().lower() != "bytes":
            return None
        first = spec.split(",")[0].strip()
        m = re.match(r"^(\d*)-(\d*)$", first)
        if not m:
            return None
        a, b = m.group(1), m.group(2)
        if a == "" and b == "":
            return None
        if a == "":
            length = int(b)
            if length <= 0:
                return "unsatisfiable"
            start, end = max(0, size - length), size - 1
        else:
            start = int(a)
            end = int(b) if b != "" else size - 1
            end = min(end, size - 1)
        if start >= size or start > end:
            return "unsatisfiable"
        return (start, end)
    except Exception:
        return None


def _iter_file(path, start, end):
    remaining = end - start + 1
    with open(path, "rb") as fh:
        fh.seek(start)
        while remaining > 0:
            chunk = fh.read(min(_CHUNK, remaining))
            if not chunk:
                break
            remaining -= len(chunk)
            yield chunk


def _file_range_response(path, request, media_type):
    path = Path(path)
    if not path.is_file():
        raise HTTPException(status_code=404, detail="file not found")
    size = path.stat().st_size
    if size == 0:
        return Response(content=b"", media_type=media_type,
                        headers={"Accept-Ranges": "bytes"})
    headers = {"Accept-Ranges": "bytes"}
    rng_header = request.headers.get("range")
    if rng_header:
        rng = _parse_range(rng_header, size)
        if rng == "unsatisfiable":
            return Response(status_code=416,
                            headers={"Content-Range": "bytes */%d" % size,
                                     "Accept-Ranges": "bytes"})
        if rng is not None:
            start, end = rng
            headers["Content-Range"] = "bytes %d-%d/%d" % (start, end, size)
            headers["Content-Length"] = str(end - start + 1)
            return StreamingResponse(_iter_file(path, start, end), status_code=206,
                                     media_type=media_type, headers=headers)
    headers["Content-Length"] = str(size)
    return StreamingResponse(_iter_file(path, 0, size - 1), status_code=200,
                             media_type=media_type, headers=headers)


def _bytes_range_response(data, request, media_type):
    size = len(data)
    headers = {"Accept-Ranges": "bytes"}
    rng_header = request.headers.get("range")
    if rng_header and size > 0:
        rng = _parse_range(rng_header, size)
        if rng == "unsatisfiable":
            return Response(status_code=416,
                            headers={"Content-Range": "bytes */%d" % size,
                                     "Accept-Ranges": "bytes"})
        if rng is not None:
            start, end = rng
            headers["Content-Range"] = "bytes %d-%d/%d" % (start, end, size)
            return Response(content=data[start:end + 1], status_code=206,
                            media_type=media_type, headers=headers)
    return Response(content=data, media_type=media_type, headers=headers)


# --------------------------------------------------------------------------
# Media helpers (frame/thumb endpoints)
# --------------------------------------------------------------------------


def _read_frame(video_path, n):
    cap = cv2.VideoCapture(str(video_path))
    if n > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, n)
    ok, frame = cap.read()
    cap.release()
    return frame if ok else None


def _resize_width(img, width):
    if not width:
        return img
    width = max(16, min(int(width), 4096))
    h, w = img.shape[:2]
    if w == width:
        return img
    nh = max(1, int(round(h * width / float(w))))
    return cv2.resize(img, (width, nh), interpolation=cv2.INTER_AREA)


def _jpeg_bytes(img):
    ok, buf = cv2.imencode(".jpg", img, [int(cv2.IMWRITE_JPEG_QUALITY), JPEG_QUALITY])
    if not ok:
        raise HTTPException(status_code=500, detail="JPEG encode failed")
    return bytes(buf.tobytes())


def _probe_video(path):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        cap.release()
        return None
    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    ok, _ = cap.read()
    if not ok or width <= 0 or height <= 0:
        cap.release()
        return None
    if fps != fps or fps <= 0:
        fps = 30.0
    if frames <= 0:
        # Rare container fallback: count frames by grabbing.
        frames = 1
        while cap.grab():
            frames += 1
    cap.release()
    return {"fps": round(float(fps), 3), "frames": int(frames),
            "width": width, "height": height,
            "duration_s": round(frames / float(fps), 3)}


def _frames_index(job):
    """Lazy frame->metadata map for the frame endpoint (internal cache)."""
    idx = job.get("_frames_index")
    if not isinstance(idx, dict) or not idx:
        meta = _read_json(JOBS_DIR / job["job_id"] / "frames.json", []) or []
        idx = {int(m["frame"]): m for m in meta}
        # Cache only a non-empty index: frames.json may not exist yet while the
        # job is still processing, and a cached {} would stick until restart.
        if idx:
            job["_frames_index"] = idx
    return idx


def _load_npz_masks(job_dir, n):
    """Return (near, {tid: mask}) for frame n, or (None, {}) when absent."""
    npz_path = Path(job_dir) / "masks" / ("f%06d.npz" % n)
    if not npz_path.is_file():
        return None, {}
    near, objs = None, {}
    with np.load(str(npz_path)) as npz:
        if "near" in npz.files:
            near = npz["near"].astype(bool)
        for key in npz.files:
            if key.startswith("obj_"):
                objs[int(key[4:])] = npz[key].astype(bool)
    return near, objs


def _draw_overlay(img, frame_meta, obj_masks):
    """Contour + '#tid label' tag for every conflicted object on the frame."""
    for inst in frame_meta.get("instances", []):
        if not inst.get("conflict"):
            continue
        tid = int(inst["track_id"])
        rgb = PALETTE[tid % len(PALETTE)]
        bgr = (int(rgb[2]), int(rgb[1]), int(rgb[0]))
        m = obj_masks.get(tid)
        if m is not None and m.shape[:2] == img.shape[:2] and m.any():
            cnts, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL,
                                       cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(img, cnts, -1, bgr, 2)
        bbox = inst.get("bbox") or [0, 0, 0, 0]
        x, y = int(bbox[0]), int(bbox[1])
        tag = "#%d %s" % (tid, inst.get("label", "object"))
        (tw, th), base = cv2.getTextSize(tag, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        ty = max(th + 4, y - 4)
        cv2.rectangle(img, (x, ty - th - 4), (x + tw + 6, ty + base), bgr, -1)
        cv2.putText(img, tag, (x + 3, ty), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                    (255, 255, 255), 1, cv2.LINE_AA)
    return img


# --------------------------------------------------------------------------
# FastAPI app + middleware
# --------------------------------------------------------------------------

app = FastAPI(title="UCAP Tool", version=VERSION)


@app.middleware("http")
async def _auth_middleware(request, call_next):
    # AUTH_TOKEN is read at call time so the notebook can assign it after import.
    token = AUTH_TOKEN
    path = request.url.path
    if token and path.startswith("/api/") and path != "/api/health":
        supplied = request.headers.get("x-ucap-token") or request.query_params.get("token")
        if supplied != token:
            return Response(content=json.dumps({"detail": "invalid token"}),
                            status_code=401, media_type="application/json")
    return await call_next(request)


# Added after the auth middleware so CORS runs first (handles preflight).
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_methods=["*"], allow_headers=["*"])


@app.on_event("startup")
def _on_startup():
    _rescan_disk()


# --------------------------------------------------------------------------
# Endpoints (SPEC 4)
# --------------------------------------------------------------------------


@app.get("/")
def index():
    candidates = []
    env = os.environ.get("UCAP_INDEX")
    if env:
        candidates.append(Path(env))
    candidates.append(Path("frontend") / "index.html")
    candidates.append(Path("index.html"))
    here = Path(__file__).resolve().parent
    candidates.append(here / "frontend" / "index.html")
    candidates.append(here / "index.html")
    for cand in candidates:
        try:
            if cand.is_file():
                return HTMLResponse(cand.read_text(encoding="utf-8"))
        except Exception:
            continue
    return HTMLResponse("<h1>UCAP Tool</h1><p>File frontend/index.html "
                        "not found.</p>", status_code=404)


@app.get("/api/health")
def api_health():
    return {"status": "ok", "gpu": _gpu_name(),
            "models": {"sam3": bool(_SAM3_CACHE), "unidepth": _UNIDEPTH is not None},
            "auth_required": bool(AUTH_TOKEN), "version": VERSION}


@app.post("/api/upload")
async def api_upload(request: Request):
    _ensure_dirs()
    try:
        form = await request.form()
    except Exception as e:
        raise HTTPException(status_code=400,
                            detail="multipart form expected: " + str(e))
    upfile = form.get("file")
    if upfile is None or not hasattr(upfile, "file"):
        raise HTTPException(status_code=400, detail="missing multipart field 'file'")
    video_id = uuid.uuid4().hex[:12]
    fname = _safe_filename(getattr(upfile, "filename", "video.mp4"))
    vdir = UPLOADS_DIR / video_id
    vdir.mkdir(parents=True, exist_ok=True)
    dst = vdir / fname
    with open(dst, "wb") as fh:
        shutil.copyfileobj(upfile.file, fh)
    probed = _probe_video(dst)
    if probed is None:
        shutil.rmtree(vdir, ignore_errors=True)
        raise HTTPException(status_code=400, detail="cv2 cannot read this file as a video")
    meta = {"video_id": video_id, "filename": fname}
    meta.update(probed)
    _write_json(vdir / "meta.json", meta)
    with _LOCK:
        VIDEOS[video_id] = meta
    return meta


@app.get("/api/videos")
def api_videos():
    return list(VIDEOS.values())


@app.get("/api/videos/{video_id}/thumb.jpg")
def api_video_thumb(video_id: str, request: Request, width: int = 480):
    if video_id not in VIDEOS:
        raise HTTPException(status_code=404, detail="unknown video_id")
    frame = _read_frame(_video_path(video_id), 0)
    if frame is None:
        raise HTTPException(status_code=404, detail="cannot read first frame")
    return _bytes_range_response(_jpeg_bytes(_resize_width(frame, width)),
                                 request, "image/jpeg")


@app.post("/api/jobs")
def api_create_job(payload: dict = Body(...)):
    _ensure_dirs()
    if not isinstance(payload, dict):
        raise HTTPException(status_code=422, detail="body must be a JSON object")
    video_id = str(payload.get("video_id", ""))
    if video_id not in VIDEOS:
        raise HTTPException(status_code=404, detail="unknown video_id")
    params = _validate_params(payload.get("params") or {})
    job_id = uuid.uuid4().hex[:12]
    job_dir = JOBS_DIR / job_id
    (job_dir / "masks").mkdir(parents=True, exist_ok=True)
    job = {"job_id": job_id, "video_id": video_id,
           "video_filename": VIDEOS[video_id]["filename"],
           "state": "queued", "created_utc": _now_iso(),
           "params": params, "progress": _default_progress(),
           "stats": None, "render_stats": None, "error": None}
    _write_json(job_dir / "params.json", params)
    with _LOCK:
        JOBS[job_id] = job
    _save_state(job)
    _enqueue("process", job_id)
    return {"job_id": job_id}


@app.get("/api/jobs")
def api_list_jobs():
    items = sorted(JOBS.values(), key=lambda j: j.get("created_utc") or "", reverse=True)
    return [{"job_id": j["job_id"], "video_id": j["video_id"],
             "video_filename": j["video_filename"], "state": j["state"],
             "created_utc": j["created_utc"], "stats": j["stats"]} for j in items]


@app.get("/api/jobs/{job_id}")
def api_job_status(job_id: str):
    return _job_status(_get_job_or_404(job_id))


@app.get("/api/jobs/{job_id}/preview.mp4")
def api_job_preview(job_id: str, request: Request):
    _get_job_or_404(job_id)
    path = JOBS_DIR / job_id / "preview.mp4"
    if not path.is_file():
        raise HTTPException(status_code=404, detail="preview not ready")
    return _file_range_response(path, request, "video/mp4")


@app.get("/api/jobs/{job_id}/conflicts")
def api_job_conflicts(job_id: str):
    job = _get_job_or_404(job_id)
    if job["state"] not in ("done", "rendering", "rendered"):
        raise HTTPException(status_code=409,
                            detail="job not processed yet (state: %s)" % job["state"])
    video = VIDEOS.get(job["video_id"], {})
    frames_meta = _read_json(JOBS_DIR / job_id / "frames.json", []) or []
    tracks = _read_json(JOBS_DIR / job_id / "tracks.json", {}) or {}
    conflicts = _collect_conflicts(frames_meta)
    return {"job_id": job_id,
            "video": {"fps": video.get("fps"), "frames": video.get("frames"),
                      "width": video.get("width"), "height": video.get("height")},
            "tracks": tracks,
            "conflicts": conflicts,
            "conflict_frames": sorted({c["frame"] for c in conflicts})}


@app.get("/api/jobs/{job_id}/frame/{n}.jpg")
def api_job_frame(job_id: str, n: int, request: Request, mode: str = "raw",
                  overlay: int = 0, width: Optional[int] = None):
    job = _get_job_or_404(job_id)
    src = _video_path(job["video_id"])
    if src is None:
        raise HTTPException(status_code=404, detail="source video not found")
    frame = _read_frame(src, max(0, int(n)))
    if frame is None:
        raise HTTPException(status_code=404, detail="frame not available")
    job_dir = JOBS_DIR / job_id
    near, obj_masks = _load_npz_masks(job_dir, int(n))
    if mode == "redacted" and obj_masks:
        nz = near if (near is not None and near.shape == frame.shape[:2]) \
            else np.zeros(frame.shape[:2], bool)
        blur = np.zeros(frame.shape[:2], bool)
        for m in obj_masks.values():
            if m.shape != frame.shape[:2]:
                continue
            blur |= (m & ~nz) if job["params"]["preserve_policy"] == "carve" else m
        frame = _apply_redaction(frame, blur, job["params"])
    if overlay:
        fm = _frames_index(job).get(int(n))
        if fm:
            frame = _draw_overlay(frame, fm, obj_masks)
    frame = _resize_width(frame, width)
    return _bytes_range_response(_jpeg_bytes(frame), request, "image/jpeg")


@app.get("/api/jobs/{job_id}/decisions")
def api_get_decisions(job_id: str):
    _get_job_or_404(job_id)
    return _read_decisions(JOBS_DIR / job_id)


@app.post("/api/jobs/{job_id}/decisions")
def api_post_decisions(job_id: str, payload: dict = Body(...)):
    _get_job_or_404(job_id)
    decisions = _validate_decisions(payload)
    with _LOCK:
        _write_json(JOBS_DIR / job_id / "decisions.json", decisions)
    return {"ok": True}


@app.post("/api/jobs/{job_id}/render")
def api_render(job_id: str, payload: Optional[dict] = Body(default=None)):
    job = _get_job_or_404(job_id)
    if job["state"] not in ("done", "rendered"):
        raise HTTPException(status_code=409,
                            detail="job not ready to render (state: %s)" % job["state"])
    if isinstance(payload, dict) and payload.get("unresolved_policy") is not None:
        pol = payload["unresolved_policy"]
        if pol not in ("failsafe", "carve"):
            raise HTTPException(status_code=422,
                                detail="unresolved_policy must be 'failsafe' or 'carve'")
        with _LOCK:
            decisions = _read_decisions(JOBS_DIR / job_id)
            decisions["unresolved_policy"] = pol
            _write_json(JOBS_DIR / job_id / "decisions.json", decisions)
    _set_state(job, "rendering")
    _enqueue("render", job_id)
    return {"ok": True}


@app.get("/api/jobs/{job_id}/final.mp4")
def api_job_final(job_id: str, request: Request):
    _get_job_or_404(job_id)
    path = JOBS_DIR / job_id / "final.mp4"
    if not path.is_file():
        raise HTTPException(status_code=404, detail="final video not ready")
    return _file_range_response(path, request, "video/mp4")


@app.get("/api/jobs/{job_id}/log")
def api_job_log(job_id: str):
    _get_job_or_404(job_id)
    log = _read_json(JOBS_DIR / job_id / "decision_log.json", None)
    if log is None:
        raise HTTPException(status_code=404, detail="decision log not ready")
    return log


# --------------------------------------------------------------------------
# Entry point
# --------------------------------------------------------------------------


def run(port=8000, host="0.0.0.0"):
    """Start uvicorn (blocking). The notebook runs this in a daemon thread."""
    import uvicorn
    uvicorn.run(app, host=host, port=int(port), log_level="info")


if __name__ == "__main__":
    run(port=int(os.environ.get("UCAP_PORT", "8000")))


In [ ]:
%%writefile frontend/index.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>UCAP Tool</title>
<style>
/* ---------- base / dark theme (no external resources) ---------- */
:root {
  --bg: #0f1318;
  --bg2: #151b22;
  --card: #171e26;
  --border: #242f3b;
  --text: #dbe2ea;
  --muted: #8b98a5;
  --accent: #4da3ff;
  --accent-dim: #2b5e9c;
  --ok: #43d17c;
  --warn: #e3b341;
  --err: #e5534b;
}
* { box-sizing: border-box; }
html, body { margin: 0; padding: 0; }
body {
  background: var(--bg);
  color: var(--text);
  font-family: "Segoe UI", system-ui, Arial, sans-serif;
  font-size: 14px;
  line-height: 1.45;
}
h1, h2, h3 { font-weight: 600; }
h2 { font-size: 15px; margin: 0 0 10px 0; }
a { color: var(--accent); }
.hidden { display: none !important; }
.muted { color: var(--muted); }

/* ---------- header ---------- */
header {
  display: flex; align-items: center; gap: 12px;
  padding: 10px 18px;
  background: var(--bg2);
  border-bottom: 1px solid var(--border);
  position: sticky; top: 0; z-index: 50;
}
.brand { font-size: 18px; font-weight: 700; letter-spacing: 0.4px; margin: 0; }
.header-spacer { flex: 1; }
.dot {
  width: 11px; height: 11px; border-radius: 50%;
  display: inline-block; flex: none;
  background: #5a6673;
}
.dot.ok { background: var(--ok); }
.dot.err { background: var(--err); }
.dot.warn { background: var(--warn); }
#health-text { color: var(--muted); font-size: 13px; }

/* ---------- buttons / inputs ---------- */
button {
  background: #1f2935; color: var(--text);
  border: 1px solid var(--border); border-radius: 7px;
  padding: 6px 12px; cursor: pointer; font-size: 13px;
  font-family: inherit;
}
button:hover { border-color: var(--accent-dim); }
button:disabled { opacity: 0.45; cursor: not-allowed; }
button.primary {
  background: var(--accent-dim); border-color: var(--accent);
  font-weight: 600;
}
button.primary:hover { background: #356fb5; }
button.big { padding: 10px 16px; font-size: 14px; width: 100%; margin-top: 10px; }
input[type="text"], input[type="number"], input[type="password"], select {
  background: #10151b; color: var(--text);
  border: 1px solid var(--border); border-radius: 6px;
  padding: 6px 8px; font-size: 13px; font-family: inherit;
}
input[type="range"] { width: 100%; accent-color: var(--accent); }
input[type="color"] { background: #10151b; border: 1px solid var(--border); border-radius: 6px; padding: 2px; width: 56px; height: 30px; }
input[type="checkbox"] { accent-color: var(--accent); }
select { max-width: 100%; }
label.check { display: flex; align-items: center; gap: 8px; margin: 8px 0; cursor: pointer; }
.lbl { display: block; margin: 10px 0 4px; color: var(--muted); font-size: 12.5px; }
.hint { color: var(--muted); font-size: 12px; margin: 4px 0 6px; }
.row { display: flex; gap: 8px; align-items: center; }
.row input[type="text"] { flex: 1; }

/* ---------- tabs ---------- */
nav.tabs {
  display: flex; gap: 4px; padding: 8px 18px 0;
  background: var(--bg2); border-bottom: 1px solid var(--border);
}
nav.tabs button {
  border-radius: 8px 8px 0 0; border-bottom: none;
  padding: 8px 18px; font-size: 14px;
  background: #131920; color: var(--muted);
}
nav.tabs button.active {
  background: var(--card); color: var(--text);
  border-color: var(--border); font-weight: 600;
  box-shadow: inset 0 2px 0 var(--accent);
}

/* ---------- cards / layout ---------- */
.card {
  background: var(--card); border: 1px solid var(--border);
  border-radius: 10px; padding: 14px; margin-bottom: 14px;
}
fieldset {
  border: 1px solid var(--border); border-radius: 8px;
  margin: 0 0 12px 0; padding: 8px 12px 12px;
}
legend { color: var(--accent); font-size: 12.5px; padding: 0 6px; }
.grid-process {
  display: grid; grid-template-columns: minmax(330px, 410px) minmax(0, 1fr);
  gap: 14px; padding: 14px 18px; align-items: start;
}
@media (max-width: 980px) { .grid-process { grid-template-columns: 1fr; } }

/* ---------- upload ---------- */
#dropzone {
  border: 2px dashed var(--border); border-radius: 10px;
  padding: 22px 12px; text-align: center; color: var(--muted);
  cursor: pointer; transition: border-color 0.15s, background 0.15s;
}
#dropzone.drag { border-color: var(--accent); background: #16202c; color: var(--text); }
.linklike { color: var(--accent); text-decoration: underline; cursor: pointer; }
.bar { background: #10151b; border: 1px solid var(--border); border-radius: 6px; height: 14px; overflow: hidden; flex: 1; }
.bar .fill { height: 100%; width: 0%; background: var(--accent); transition: width 0.2s; }
#upload-progress { display: flex; gap: 8px; align-items: center; margin-top: 8px; }
#video-meta { display: flex; gap: 10px; margin-top: 10px; align-items: flex-start; }
#video-thumb { width: 150px; border-radius: 8px; border: 1px solid var(--border); }
#video-info { font-size: 12.5px; color: var(--muted); }
#video-info b { color: var(--text); }

/* ---------- chips ---------- */
.chips { display: flex; flex-wrap: wrap; gap: 6px; margin: 6px 0; }
.chip {
  display: inline-flex; align-items: center; gap: 6px;
  background: #1f2935; border: 1px solid var(--border);
  border-radius: 999px; padding: 3px 6px 3px 10px; font-size: 12.5px;
}
.chip-x {
  border: none; background: transparent; color: var(--muted);
  padding: 0 4px; font-size: 12px; line-height: 1;
}
.chip-x:hover { color: var(--err); }

/* ---------- job status / stats ---------- */
.progress-info { display: flex; gap: 14px; flex-wrap: wrap; color: var(--muted); font-size: 12.5px; margin-top: 6px; }
.stats-grid { display: grid; grid-template-columns: repeat(auto-fill, minmax(118px, 1fr)); gap: 8px; margin: 10px 0; }
.stat-card { background: #10151b; border: 1px solid var(--border); border-radius: 8px; padding: 8px 10px; }
.stat-card .v { font-size: 17px; font-weight: 700; }
.stat-card .k { font-size: 11.5px; color: var(--muted); }
video {
  display: block; width: 100%; max-height: 68vh; object-fit: contain;
  border-radius: 8px; background: #000; margin-top: 8px;
}
.warn-text { color: var(--warn); font-size: 12.5px; margin-top: 6px; }
.err-text { color: var(--err); font-size: 13px; margin-top: 6px; }

/* ---------- job list ---------- */
.job-row {
  display: flex; align-items: center; gap: 10px;
  border: 1px solid var(--border); border-radius: 8px;
  padding: 8px 10px; margin-bottom: 6px; font-size: 12.5px;
}
.job-row .jr-main { flex: 1; min-width: 0; }
.job-row .jr-name { font-weight: 600; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; }
.job-row .jr-sub { color: var(--muted); }
.badge {
  display: inline-block; border-radius: 999px; padding: 1px 9px;
  font-size: 11.5px; border: 1px solid var(--border); background: #1f2935;
  white-space: nowrap;
}
.badge.ok { color: var(--ok); border-color: #2c5a40; }
.badge.err { color: var(--err); border-color: #6b2f2c; }
.badge.run { color: var(--accent); border-color: var(--accent-dim); }
.badge.unres { color: var(--warn); border-color: #6b5a23; }

/* ---------- settings popover ---------- */
.popover {
  position: absolute; top: 50px; right: 18px;
  background: var(--card); border: 1px solid var(--border);
  border-radius: 10px; padding: 14px; width: 320px; z-index: 100;
  box-shadow: 0 8px 28px rgba(0,0,0,0.5);
}
.popover label { display: block; margin-bottom: 10px; font-size: 12.5px; color: var(--muted); }
.popover input { width: 100%; margin-top: 4px; }

/* ---------- conflicts tab ---------- */
.summary-bar {
  display: flex; align-items: center; gap: 16px; flex-wrap: wrap;
  padding: 10px 18px; background: var(--bg2);
  border-bottom: 1px solid var(--border); font-size: 13px;
}
.summary-bar .check { margin: 0; }
.saved-ind { color: var(--ok); font-size: 12.5px; min-width: 90px; }
.saved-ind.dirty { color: var(--warn); }
.saved-ind.err { color: var(--err); }
.cf-grid {
  display: grid; grid-template-columns: 280px minmax(0, 1fr) 300px;
  gap: 14px; padding: 14px 18px; align-items: start;
}
@media (max-width: 1150px) { .cf-grid { grid-template-columns: 240px minmax(0, 1fr); } #cf-render { grid-column: 1 / -1; } }
@media (max-width: 850px) { .cf-grid { grid-template-columns: 1fr; } }

.cdot { width: 11px; height: 11px; border-radius: 50%; display: inline-block; flex: none; }
.track-row { border: 1px solid var(--border); border-radius: 8px; padding: 8px 10px; margin-bottom: 8px; }
.track-head { display: flex; align-items: center; gap: 8px; flex-wrap: wrap; }
.track-sub { color: var(--muted); font-size: 12px; margin: 4px 0 6px; }
.track-btns { display: flex; gap: 6px; flex-wrap: wrap; }

.viewer-top { display: flex; align-items: center; gap: 10px; flex-wrap: wrap; margin-bottom: 8px; }
.viewer-label { font-size: 13px; font-weight: 600; }
#cf-frame-img {
  width: 100%; max-height: 56vh; object-fit: contain;
  border-radius: 8px; background: #000; display: block;
}
.seg { display: inline-flex; border: 1px solid var(--border); border-radius: 7px; overflow: hidden; }
.seg button { border: none; border-radius: 0; }
.seg button.active { background: var(--accent-dim); color: #fff; }
#cf-strip {
  display: flex; gap: 4px; flex-wrap: wrap; margin-top: 8px;
  max-height: 84px; overflow-y: auto;
}
#cf-strip button { padding: 2px 8px; font-size: 11.5px; border-radius: 5px; }
#cf-strip button.current { background: var(--accent-dim); border-color: var(--accent); color: #fff; }
#cf-strip button.unres { box-shadow: inset 0 -2px 0 var(--warn); }

#cf-cards { display: grid; grid-template-columns: repeat(auto-fill, minmax(300px, 1fr)); gap: 10px; margin-top: 12px; }
.conflict-card {
  background: var(--card); border: 1px solid var(--border);
  border-left: 4px solid var(--muted); border-radius: 8px;
  padding: 10px 12px; cursor: pointer;
}
.conflict-card.selected { outline: 2px solid var(--accent); }
.cc-head { display: flex; align-items: center; gap: 8px; flex-wrap: wrap; }
.cc-info { color: var(--muted); font-size: 12px; margin: 5px 0 8px; }
.cc-opts { display: flex; flex-direction: column; gap: 4px; }
.opt-btn { text-align: left; font-size: 12.5px; padding: 5px 9px; }
.opt-btn.active { background: var(--accent-dim); border-color: var(--accent); color: #fff; }
.opt-btn.small { font-size: 12px; padding: 3px 8px; }
.opt-num { display: inline-block; width: 16px; color: var(--accent); font-weight: 700; }
.opt-btn.active .opt-num { color: #fff; }
.auto-adv { margin-top: 10px; color: var(--muted); }
.kbd-hint { color: var(--muted); font-size: 11.5px; margin-top: 6px; }

#render-warning { font-size: 12.5px; margin-bottom: 8px; }
#render-warning.warn { color: var(--warn); }
#render-warning.ok { color: var(--ok); }
.dl-links { display: flex; flex-direction: column; gap: 6px; margin-top: 8px; }

/* ---------- toast ---------- */
.toast {
  position: fixed; bottom: 22px; left: 50%; transform: translateX(-50%);
  background: #1f2935; border: 1px solid var(--accent-dim);
  color: var(--text); padding: 10px 18px; border-radius: 9px;
  font-size: 13px; z-index: 200; box-shadow: 0 6px 24px rgba(0,0,0,0.5);
  max-width: 80vw;
}
</style>
</head>
<body>

<header>
  <h1 class="brand">UCAP Tool</h1>
  <span class="muted">anonymization with interaction-zone carve-out</span>
  <div class="header-spacer"></div>
  <span id="health-dot" class="dot" title="API status"></span>
  <span id="health-text">API: checking...</span>
  <button id="btn-settings" type="button">Settings</button>
  <div id="settings-pop" class="popover hidden">
    <label>API URL
      <input id="set-api" type="text" placeholder="http://localhost:8000">
    </label>
    <label>Token (optional, X-UCAP-Token header)
      <input id="set-token" type="password" placeholder="empty = no token">
    </label>
    <button id="set-save" class="primary" type="button">Save settings</button>
  </div>
</header>

<nav class="tabs">
  <button id="tab-btn-process" class="active" type="button">Processing</button>
  <button id="tab-btn-conflicts" type="button">Conflicts</button>
</nav>

<main>

<!-- ================= TAB 1: PROCESSING ================= -->
<section id="tab-process">
  <div class="grid-process">
    <div><!-- left column: video + params -->
      <div class="card">
        <h2>Video</h2>
        <div id="dropzone">
          Drag and drop a video file here<br>
          or <span class="linklike">choose a file from disk</span>
          <input id="file-input" type="file" accept="video/*" class="hidden">
        </div>
        <div id="upload-progress" class="hidden">
          <div class="bar"><div class="fill" id="upload-fill"></div></div>
          <span id="upload-pct" class="muted">0%</span>
        </div>
        <label class="lbl">Previously uploaded videos</label>
        <select id="video-select" style="width:100%">
          <option value="">- select a video -</option>
        </select>
        <div id="video-meta" class="hidden">
          <img id="video-thumb" alt="video thumbnail">
          <div id="video-info"></div>
        </div>
      </div>

      <div class="card">
        <h2>Parameters</h2>

        <fieldset>
          <legend>Detection (SAM3)</legend>
          <label class="lbl">Prompts (what to redact)</label>
          <div id="prompt-chips" class="chips"></div>
          <div class="row">
            <input id="prompt-input" type="text" placeholder="e.g. phone screen">
            <button id="prompt-add" type="button">Add</button>
          </div>
          <label class="lbl">Detection threshold (score_threshold): <b id="p-score-val"></b></label>
          <input id="p-score" type="range" min="0.05" max="0.95" step="0.05">
        </fieldset>

        <fieldset>
          <legend>Depth (UniDepth)</legend>
          <label class="check"><input id="p-use-metric" type="checkbox"> metric threshold (in meters)</label>
          <div id="row-near-meters">
            <label class="lbl">Near zone (meters): <b id="p-near-meters-val"></b></label>
            <input id="p-near-meters" type="range" min="0.2" max="3" step="0.1">
          </div>
          <div id="row-near-percent" class="hidden">
            <label class="lbl">Near zone (nearest % of pixels): <b id="p-near-percent-val"></b></label>
            <input id="p-near-percent" type="range" min="5" max="95" step="5">
          </div>
          <label class="lbl">Depth every n-th frame (depth_stride): <b id="p-depth-stride-val"></b></label>
          <input id="p-depth-stride" type="range" min="1" max="5" step="1">
          <label class="lbl">UniDepth resolution level (0-9): <b id="p-res-level-val"></b></label>
          <input id="p-res-level" type="range" min="0" max="9" step="1">
        </fieldset>

        <fieldset>
          <legend>Policy and review</legend>
          <label class="lbl">Preserve policy (preserve_policy)</label>
          <select id="p-policy" style="width:100%">
            <option value="carve">carve</option>
            <option value="failsafe">failsafe</option>
          </select>
          <div id="policy-expl" class="hint"></div>
          <label class="lbl">Conflict threshold (overlap_threshold): <b id="p-overlap-val"></b></label>
          <input id="p-overlap" type="range" min="0.05" max="0.6" step="0.05">
          <label class="check"><input id="p-mark" type="checkbox"> mark conflicts in the preview (red contour)</label>
        </fieldset>

        <fieldset>
          <legend>Redaction</legend>
          <label class="lbl">Redaction type</label>
          <select id="p-redaction" style="width:100%">
            <option value="blur">blur (gaussian blur)</option>
            <option value="pixelate">pixelate (mosaic)</option>
            <option value="fill">fill (solid color)</option>
          </select>
          <div id="row-blur">
            <label class="lbl">Blur kernel size (odd number)</label>
            <input id="p-blur-ksize" type="number" min="3" max="199" step="2" style="width:110px">
          </div>
          <div id="row-pixelate" class="hidden">
            <label class="lbl">Number of pixelation blocks</label>
            <input id="p-pixelate" type="number" min="2" max="64" step="1" style="width:110px">
          </div>
          <div id="row-fill" class="hidden">
            <label class="lbl">Fill color</label>
            <input id="p-fill" type="color">
          </div>
        </fieldset>

        <fieldset>
          <legend>Range</legend>
          <label class="lbl">Maximum number of frames (empty = whole clip)</label>
          <input id="p-max-frames" type="number" min="1" placeholder="whole clip" style="width:140px">
        </fieldset>

        <button id="btn-process" class="primary big" type="button">Process</button>
      </div>
    </div>

    <div><!-- right column: job status + preview + job list -->
      <div class="card">
        <h2>Job status</h2>
        <div id="job-none" class="muted">No active job. Select a video, set the parameters and click "Process".</div>
        <div id="job-status" class="hidden">
          <div class="row" style="margin-bottom:6px">
            <span id="job-state-badge" class="badge run"></span>
            <span id="job-id-label" class="muted" style="font-size:12px"></span>
          </div>
          <div id="job-progress-wrap">
            <div class="bar"><div class="fill" id="job-fill"></div></div>
            <div class="progress-info">
              <span id="pi-frames"></span>
              <span id="pi-fps"></span>
              <span id="pi-eta"></span>
              <span id="pi-stage"></span>
            </div>
          </div>
          <div id="job-error" class="err-text hidden"></div>
          <div id="job-warnings" class="warn-text hidden"></div>
          <div id="stats-cards" class="stats-grid hidden"></div>
          <video id="preview-video" controls preload="metadata" class="hidden"></video>
          <button id="btn-goto-conflicts" class="primary big hidden" type="button">Go to conflicts</button>
        </div>
      </div>

      <div class="card">
        <div class="row" style="justify-content:space-between">
          <h2 style="margin:0">Jobs</h2>
          <button id="btn-refresh-jobs" type="button">Refresh</button>
        </div>
        <div id="job-list" style="margin-top:10px">
          <div class="muted">No jobs.</div>
        </div>
      </div>
    </div>
  </div>
</section>

<!-- ================= TAB 2: CONFLICTS ================= -->
<section id="tab-conflicts" class="hidden">
  <div class="summary-bar">
    <b id="cf-counts">Conflicts: 0 | Resolved: 0 | Remaining: 0</b>
    <label class="check"><input type="checkbox" id="cf-only-unresolved"> unresolved only</label>
    <label class="check">Job:
      <select id="cf-job-select"><option value="">- select -</option></select>
    </label>
    <div class="header-spacer"></div>
    <span id="cf-saved" class="saved-ind"></span>
  </div>

  <div id="cf-empty" class="card" style="margin:14px 18px">
    Select a completed job from the list above to review conflicts.
  </div>

  <div id="cf-body" class="cf-grid hidden">

    <aside id="cf-tracks" class="card" style="margin-bottom:0">
      <h2>Objects</h2>
      <div id="track-list"></div>
    </aside>

    <div id="cf-center">
      <div class="card" style="margin-bottom:0">
        <div class="viewer-top">
          <div class="seg">
            <button id="mode-raw" type="button" class="active">original</button>
            <button id="mode-red" type="button">redacted</button>
          </div>
          <button id="btn-prev-frame" type="button">&#8592; previous</button>
          <button id="btn-next-frame" type="button">next &#8594;</button>
          <span id="frame-label" class="viewer-label"></span>
        </div>
        <img id="cf-frame-img" alt="conflict frame preview">
        <div id="cf-strip"></div>
      </div>
      <div id="cf-cards"></div>
      <label class="check auto-adv">
        <input type="checkbox" id="cf-auto-advance" checked>
        after a decision go to the next unresolved conflict
      </label>
      <div class="kbd-hint">Shortcuts: left/right arrows = conflict frames, Tab = next card, keys 1-5 = decision for the selected card (pressing the same key again clears the decision).</div>
    </div>

    <aside id="cf-render" class="card" style="margin-bottom:0">
      <h2>Final render</h2>
      <div id="render-warning"></div>
      <label class="lbl">Policy for unresolved conflicts</label>
      <select id="render-policy" style="width:100%">
        <option value="failsafe">failsafe - blur unresolved</option>
        <option value="carve">carve - keep sharp in the near zone</option>
      </select>
      <button id="btn-render" class="primary big" type="button">Render final video</button>
      <div id="render-status" class="hint"></div>
      <div id="render-stats" class="hint"></div>
      <video id="final-video" controls preload="metadata" class="hidden"></video>
      <div id="render-links" class="dl-links hidden">
        <a id="dl-final" download="final.mp4">Download final.mp4</a>
        <a id="dl-log" download="decision_log.json">Download decision_log.json</a>
      </div>
    </aside>

  </div>
</section>

</main>

<div id="toast" class="toast hidden"></div>

<script>
"use strict";
/* =========================================================================
 * UCAP Tool frontend - single file, vanilla JS, no external resources.
 * Structure: one `state` object + small render functions + api() helper.
 * All UI text is English (per SPEC section 9); code and comments are English.
 * ========================================================================= */

/* ---------- constants ---------- */

// Shared track palette - MUST match the server palette (SPEC 4.4), RGB order.
const PALETTE = [
  [244, 67, 54], [33, 150, 243], [76, 175, 80], [255, 193, 7],
  [156, 39, 176], [0, 188, 212], [255, 87, 34], [205, 220, 57],
];

// The 5 decision options, in keyboard order 1..5 (SPEC sections 3 + 9).
const OPTION_KEYS = ["unblur_track", "unblur_frame", "blur_frame", "blur_track", "drop_frame"];
const OPTION_LABELS = {
  unblur_track: "Don't blur - whole object (all frames)",
  unblur_frame: "Don't blur - this frame only",
  blur_frame: "Blur - this frame only",
  blur_track: "Blur - whole object (all frames)",
  drop_frame: "Remove frame",
};
const DECISION_SHORT = {
  unblur_track: "don't blur (whole object)",
  unblur_frame: "don't blur (this frame)",
  blur_frame: "blur (this frame)",
  blur_track: "blur (whole object)",
};
const STAGE_LABELS = {
  pipeline: "analysis (SAM3 + depth)",
  encode: "H.264 encoding",
  render: "final render",
};
const STATE_LABELS = {
  queued: "queued", processing: "processing", done: "done",
  error: "error", rendering: "rendering", rendered: "rendered",
};

const DEFAULT_PARAMS = {
  prompts: ["face", "licence plate"],
  score_threshold: 0.6,
  near_meters: 1.1,
  use_metric: true,
  near_percent: 40,
  overlap_threshold: 0.15,
  preserve_policy: "carve",
  redaction: "blur",
  blur_ksize: 41,
  pixelate_blocks: 16,
  fill_bgr: [255, 0, 0],
  max_frames: null,
  depth_stride: 1,
  unidepth_res_level: 9,
  mark_conflicts_in_preview: true,
};

const LS_SETTINGS = "ucap_settings";
const LS_LAST_JOB = "ucap_last_job";

/* ---------- state ---------- */

function defaultApiUrl() {
  const o = window.location.origin;
  return (o && o.indexOf("http") === 0) ? o : "http://localhost:8000";
}

function emptyDecisions() {
  return { tracks: {}, frames: {}, dropped_frames: [], unresolved_policy: "failsafe" };
}

const state = {
  settings: { apiUrl: defaultApiUrl(), token: "" },
  health: null,
  tab: "process",
  videos: [],
  videoId: null,
  params: JSON.parse(JSON.stringify(DEFAULT_PARAMS)),
  jobId: null,
  job: null,
  jobs: [],
  pollTimer: null,
  cf: {
    jobId: null,
    job: null,
    data: null,          // conflicts payload (SPEC 4.3)
    decisions: emptyDecisions(),
    frame: null,         // current frame NUMBER (not index)
    mode: "raw",
    selIdx: 0,           // selected conflict card index on the current frame
    onlyUnresolved: false,
    autoAdvance: true,
    dirty: false,
    saveTimer: null,
    renderTimer: null,
  },
};

/* ---------- tiny DOM helpers ---------- */

function $(sel) { return document.querySelector(sel); }

function el(tag, attrs, ...children) {
  const node = document.createElement(tag);
  attrs = attrs || {};
  for (const k of Object.keys(attrs)) {
    const v = attrs[k];
    if (v === null || v === undefined) continue;
    if (k === "class") node.className = v;
    else if (k.indexOf("on") === 0) node.addEventListener(k.slice(2), v);
    else node.setAttribute(k, v);
  }
  const flat = [];
  (function add(list) {
    for (const c of list) {
      if (c === null || c === undefined) continue;
      if (Array.isArray(c)) add(c); else flat.push(c);
    }
  })(children);
  for (const c of flat) {
    node.append(c.nodeType ? c : document.createTextNode(String(c)));
  }
  return node;
}

let toastTimer = null;
function toast(msg) {
  const t = $("#toast");
  t.textContent = msg;
  t.classList.remove("hidden");
  clearTimeout(toastTimer);
  toastTimer = setTimeout(() => t.classList.add("hidden"), 2800);
}

function trackColor(tid) {
  const i = ((tid % PALETTE.length) + PALETTE.length) % PALETTE.length;
  const c = PALETTE[i];
  return "rgb(" + c[0] + "," + c[1] + "," + c[2] + ")";
}

function fmtEta(s) {
  if (s === null || s === undefined || !isFinite(s)) return "-";
  s = Math.max(0, Math.round(s));
  const m = Math.floor(s / 60), r = s % 60;
  return m + ":" + String(r).padStart(2, "0");
}

function fmtDate(iso) {
  try { return new Date(iso).toLocaleString("en-GB"); } catch (e) { return iso || ""; }
}

/* ---------- settings + api helpers ---------- */

function loadSettings() {
  try {
    const raw = localStorage.getItem(LS_SETTINGS);
    if (raw) {
      const s = JSON.parse(raw);
      if (s.apiUrl) state.settings.apiUrl = s.apiUrl;
      if (typeof s.token === "string") state.settings.token = s.token;
    }
  } catch (e) { /* ignore broken localStorage */ }
}

function saveSettings() {
  localStorage.setItem(LS_SETTINGS, JSON.stringify(state.settings));
}

function apiBase() {
  return state.settings.apiUrl.replace(/\/+$/, "");
}

// fetch helper: base URL + token header + JSON in/out + error -> Error(message)
async function api(path, opts) {
  opts = opts || {};
  const headers = Object.assign({}, opts.headers || {});
  if (state.settings.token) headers["X-UCAP-Token"] = state.settings.token;
  let body = opts.body;
  if (body !== undefined && body !== null && !(body instanceof FormData) && typeof body !== "string") {
    body = JSON.stringify(body);
    headers["Content-Type"] = "application/json";
  }
  const res = await fetch(apiBase() + path, {
    method: opts.method || "GET",
    headers: headers,
    body: body,
  });
  if (!res.ok) {
    let msg = "HTTP " + res.status;
    try {
      const j = await res.json();
      if (j && j.detail) msg += ": " + (typeof j.detail === "string" ? j.detail : JSON.stringify(j.detail));
    } catch (e) { /* non-JSON error body */ }
    throw new Error(msg);
  }
  return res.json();
}

// media URL builder: token goes into the query string because <img>/<video>
// tags cannot send headers (SPEC section 4).
function mediaUrl(path, params) {
  const q = new URLSearchParams();
  if (params) {
    for (const k of Object.keys(params)) {
      if (params[k] !== null && params[k] !== undefined) q.set(k, params[k]);
    }
  }
  if (state.settings.token) q.set("token", state.settings.token);
  const qs = q.toString();
  return apiBase() + path + (qs ? "?" + qs : "");
}

/* ---------- header: health + settings popover ---------- */

async function pollHealth() {
  const dot = $("#health-dot"), txt = $("#health-text");
  try {
    const h = await api("/api/health");
    state.health = h;
    if (h.auth_required && !state.settings.token) {
      dot.className = "dot warn";
      txt.textContent = "API: token required";
      dot.title = "The server requires a token - set it in Settings.";
    } else {
      dot.className = "dot ok";
      txt.textContent = "API: ok" + (h.gpu ? " (" + h.gpu + ")" : "");
      dot.title = "models: sam3 " + (h.models && h.models.sam3 ? "ok" : "missing") +
                  ", unidepth " + (h.models && h.models.unidepth ? "ok" : "missing");
    }
  } catch (e) {
    state.health = null;
    dot.className = "dot err";
    txt.textContent = "API: no connection";
    dot.title = String(e.message || e);
  }
}

function bindHeader() {
  const pop = $("#settings-pop");
  $("#btn-settings").addEventListener("click", (ev) => {
    ev.stopPropagation();
    $("#set-api").value = state.settings.apiUrl;
    $("#set-token").value = state.settings.token;
    pop.classList.toggle("hidden");
  });
  pop.addEventListener("click", (ev) => ev.stopPropagation());
  document.addEventListener("click", () => pop.classList.add("hidden"));
  $("#set-save").addEventListener("click", () => {
    state.settings.apiUrl = $("#set-api").value.trim() || defaultApiUrl();
    state.settings.token = $("#set-token").value.trim();
    saveSettings();
    pop.classList.add("hidden");
    toast("Settings saved");
    pollHealth();
    refreshVideos();
    refreshJobs();
    renderAll();
  });
}

/* ---------- tabs ---------- */

function switchTab(name) {
  if (state.tab === "conflicts" && name !== "conflicts" && state.cf.dirty) {
    saveDecisionsNow(); // save decisions on tab switch (SPEC 5 / tab 2)
  }
  state.tab = name;
  $("#tab-process").classList.toggle("hidden", name !== "process");
  $("#tab-conflicts").classList.toggle("hidden", name !== "conflicts");
  $("#tab-btn-process").classList.toggle("active", name === "process");
  $("#tab-btn-conflicts").classList.toggle("active", name === "conflicts");
  if (name === "conflicts") {
    refreshJobs();
    if (!state.cf.jobId) {
      const last = state.jobId || localStorage.getItem(LS_LAST_JOB);
      if (last) openConflicts(last);
    }
  }
}

function bindTabs() {
  $("#tab-btn-process").addEventListener("click", () => switchTab("process"));
  $("#tab-btn-conflicts").addEventListener("click", () => switchTab("conflicts"));
}

/* =========================================================================
 * TAB 1 - Processing
 * ========================================================================= */

/* ---------- videos + upload ---------- */

async function refreshVideos() {
  try {
    state.videos = await api("/api/videos");
  } catch (e) { return; }
  const sel = $("#video-select");
  const cur = state.videoId || "";
  sel.innerHTML = "";
  sel.append(el("option", { value: "" }, "- select a video -"));
  for (const v of state.videos) {
    sel.append(el("option", { value: v.video_id },
      v.filename + " (" + v.width + "x" + v.height + ", " + v.frames + " frames)"));
  }
  sel.value = cur;
  renderVideoMeta();
}

function selectVideo(videoId) {
  state.videoId = videoId || null;
  $("#video-select").value = videoId || "";
  renderVideoMeta();
}

function renderVideoMeta() {
  const box = $("#video-meta");
  const v = state.videos.find((x) => x.video_id === state.videoId);
  if (!v) { box.classList.add("hidden"); return; }
  box.classList.remove("hidden");
  $("#video-thumb").src = mediaUrl("/api/videos/" + v.video_id + "/thumb.jpg", { width: 480 });
  const info = $("#video-info");
  info.innerHTML = "";
  info.append(
    el("div", {}, el("b", {}, v.filename)),
    el("div", {}, "resolution: " + v.width + " x " + v.height),
    el("div", {}, "fps: " + Number(v.fps).toFixed(2) + ", frames: " + v.frames),
    el("div", {}, "duration: " + Number(v.duration_s).toFixed(1) + " s"),
  );
}

function setUploadProgress(frac) {
  const wrap = $("#upload-progress");
  if (frac === null) { wrap.classList.add("hidden"); return; }
  wrap.classList.remove("hidden");
  const pct = Math.round(frac * 100);
  $("#upload-fill").style.width = pct + "%";
  $("#upload-pct").textContent = pct + "%";
}

// XHR upload (fetch has no upload progress events)
function uploadFile(file) {
  if (!file) return;
  const xhr = new XMLHttpRequest();
  xhr.open("POST", apiBase() + "/api/upload");
  if (state.settings.token) xhr.setRequestHeader("X-UCAP-Token", state.settings.token);
  xhr.upload.onprogress = (e) => {
    if (e.lengthComputable) setUploadProgress(e.loaded / e.total);
  };
  xhr.onload = () => {
    setUploadProgress(null);
    if (xhr.status >= 200 && xhr.status < 300) {
      let meta;
      try { meta = JSON.parse(xhr.responseText); }
      catch (e) { toast("Error: invalid server response"); return; }
      if (!state.videos.some((v) => v.video_id === meta.video_id)) state.videos.push(meta);
      toast("Uploaded: " + meta.filename);
      refreshVideos().then(() => selectVideo(meta.video_id));
    } else {
      let msg = "HTTP " + xhr.status;
      try { const j = JSON.parse(xhr.responseText); if (j.detail) msg += ": " + j.detail; } catch (e) {}
      toast("Upload error: " + msg);
    }
  };
  xhr.onerror = () => { setUploadProgress(null); toast("Network error during upload"); };
  setUploadProgress(0);
  const fd = new FormData();
  fd.append("file", file);
  xhr.send(fd);
}

function bindUpload() {
  const dz = $("#dropzone");
  const fi = $("#file-input");
  dz.addEventListener("click", () => fi.click());
  fi.addEventListener("change", () => { uploadFile(fi.files[0]); fi.value = ""; });
  dz.addEventListener("dragover", (e) => { e.preventDefault(); dz.classList.add("drag"); });
  dz.addEventListener("dragleave", () => dz.classList.remove("drag"));
  dz.addEventListener("drop", (e) => {
    e.preventDefault();
    dz.classList.remove("drag");
    if (e.dataTransfer.files && e.dataTransfer.files.length) uploadFile(e.dataTransfer.files[0]);
  });
  $("#video-select").addEventListener("change", (e) => selectVideo(e.target.value));
}

/* ---------- params form ---------- */

function bgrToHex(bgr) {
  const b = bgr[0], g = bgr[1], r = bgr[2];
  return "#" + [r, g, b].map((v) => Math.max(0, Math.min(255, v)).toString(16).padStart(2, "0")).join("");
}
function hexToBgr(hex) {
  const r = parseInt(hex.slice(1, 3), 16), g = parseInt(hex.slice(3, 5), 16), b = parseInt(hex.slice(5, 7), 16);
  return [b, g, r];
}

function renderPrompts() {
  const box = $("#prompt-chips");
  box.innerHTML = "";
  state.params.prompts.forEach((p, i) => {
    box.append(el("span", { class: "chip" }, p,
      el("button", {
        class: "chip-x", type: "button", title: "Remove prompt",
        onclick: () => { state.params.prompts.splice(i, 1); renderPrompts(); },
      }, "×")));
  });
}

function addPrompt() {
  const inp = $("#prompt-input");
  const v = inp.value.trim();
  if (!v) return;
  if (!state.params.prompts.includes(v)) state.params.prompts.push(v);
  inp.value = "";
  renderPrompts();
}

function updatePolicyExpl() {
  const p = state.params.preserve_policy;
  $("#policy-expl").textContent = (p === "carve")
    ? "carve: blur privacy OUTSIDE the near zone - preserves training value, conflicts go to human review."
    : "failsafe: blur all privacy regions - conflicts are only marked for review.";
}

function updateDepthRows() {
  $("#row-near-meters").classList.toggle("hidden", !state.params.use_metric);
  $("#row-near-percent").classList.toggle("hidden", state.params.use_metric);
}

function updateRedactionRows() {
  const r = state.params.redaction;
  $("#row-blur").classList.toggle("hidden", r !== "blur");
  $("#row-pixelate").classList.toggle("hidden", r !== "pixelate");
  $("#row-fill").classList.toggle("hidden", r !== "fill");
}

// wire a range input to state.params[key] + a value label
function bindRange(inputId, valId, key, fmt) {
  const inp = $(inputId), val = $(valId);
  const show = () => { val.textContent = fmt ? fmt(state.params[key]) : String(state.params[key]); };
  inp.value = state.params[key];
  show();
  inp.addEventListener("input", () => {
    state.params[key] = parseFloat(inp.value);
    show();
  });
}

function bindParamsForm() {
  $("#prompt-add").addEventListener("click", addPrompt);
  $("#prompt-input").addEventListener("keydown", (e) => {
    if (e.key === "Enter") { e.preventDefault(); addPrompt(); }
  });
  renderPrompts();

  bindRange("#p-score", "#p-score-val", "score_threshold", (v) => v.toFixed(2));
  bindRange("#p-near-meters", "#p-near-meters-val", "near_meters", (v) => v.toFixed(1) + " m");
  bindRange("#p-near-percent", "#p-near-percent-val", "near_percent", (v) => v + "%");
  bindRange("#p-depth-stride", "#p-depth-stride-val", "depth_stride", (v) => String(v));
  bindRange("#p-res-level", "#p-res-level-val", "unidepth_res_level", (v) => String(v));
  bindRange("#p-overlap", "#p-overlap-val", "overlap_threshold", (v) => v.toFixed(2));

  const um = $("#p-use-metric");
  um.checked = state.params.use_metric;
  um.addEventListener("change", () => { state.params.use_metric = um.checked; updateDepthRows(); });
  updateDepthRows();

  const pol = $("#p-policy");
  pol.value = state.params.preserve_policy;
  pol.addEventListener("change", () => { state.params.preserve_policy = pol.value; updatePolicyExpl(); });
  updatePolicyExpl();

  const mark = $("#p-mark");
  mark.checked = state.params.mark_conflicts_in_preview;
  mark.addEventListener("change", () => { state.params.mark_conflicts_in_preview = mark.checked; });

  const red = $("#p-redaction");
  red.value = state.params.redaction;
  red.addEventListener("change", () => { state.params.redaction = red.value; updateRedactionRows(); });
  updateRedactionRows();

  const bk = $("#p-blur-ksize");
  bk.value = state.params.blur_ksize;
  bk.addEventListener("change", () => {
    let v = parseInt(bk.value, 10);
    if (!isFinite(v) || v < 3) v = 3;
    if (v % 2 === 0) v += 1; // Gaussian kernel must be odd
    state.params.blur_ksize = v;
    bk.value = v;
  });

  const px = $("#p-pixelate");
  px.value = state.params.pixelate_blocks;
  px.addEventListener("change", () => {
    const v = parseInt(px.value, 10);
    state.params.pixelate_blocks = isFinite(v) && v >= 2 ? v : 16;
    px.value = state.params.pixelate_blocks;
  });

  const fill = $("#p-fill");
  fill.value = bgrToHex(state.params.fill_bgr);
  fill.addEventListener("change", () => { state.params.fill_bgr = hexToBgr(fill.value); });

  const mf = $("#p-max-frames");
  mf.addEventListener("change", () => {
    const v = parseInt(mf.value, 10);
    state.params.max_frames = (isFinite(v) && v > 0) ? v : null;
    if (state.params.max_frames === null) mf.value = "";
  });
}

function collectParams() {
  // state.params is kept in sync by the form bindings; copy it for the POST
  return JSON.parse(JSON.stringify(state.params));
}

/* ---------- job start + polling ---------- */

async function startJob() {
  if (!state.videoId) { toast("First select or upload a video."); return; }
  if (!state.params.prompts.length) { toast("Add at least one prompt."); return; }
  const btn = $("#btn-process");
  btn.disabled = true;
  try {
    const r = await api("/api/jobs", { method: "POST", body: { video_id: state.videoId, params: collectParams() } });
    state.jobId = r.job_id;
    state.job = null;
    localStorage.setItem(LS_LAST_JOB, state.jobId);
    toast("Job started: " + state.jobId);
    startJobPoll();
    refreshJobs();
  } catch (e) {
    toast("Error starting job: " + e.message);
  }
  btn.disabled = false;
}

function stopJobPoll() {
  if (state.pollTimer) { clearInterval(state.pollTimer); state.pollTimer = null; }
}

function startJobPoll() {
  stopJobPoll();
  state.pollTimer = setInterval(pollJobOnce, 1500);
  pollJobOnce();
}

async function pollJobOnce() {
  if (!state.jobId) { stopJobPoll(); return; }
  let j;
  try {
    j = await api("/api/jobs/" + state.jobId);
  } catch (e) {
    return; // transient network error: keep polling
  }
  state.job = j;
  renderJobStatus();
  if (j.state === "done" || j.state === "rendered" || j.state === "error") {
    stopJobPoll();
    refreshJobs();
  }
}

function openJob(jobId) {
  state.jobId = jobId;
  state.job = null;
  localStorage.setItem(LS_LAST_JOB, jobId);
  startJobPoll(); // poll stops by itself on terminal states
}

function renderJobStatus() {
  const j = state.job;
  $("#job-none").classList.toggle("hidden", !!j);
  $("#job-status").classList.toggle("hidden", !j);
  if (!j) return;

  const badge = $("#job-state-badge");
  badge.textContent = STATE_LABELS[j.state] || j.state;
  badge.className = "badge " + (j.state === "error" ? "err" :
    (j.state === "done" || j.state === "rendered") ? "ok" : "run");
  $("#job-id-label").textContent = "job " + j.job_id + " | " + (j.video_filename || "");

  // progress
  const pr = j.progress || {};
  const running = (j.state === "queued" || j.state === "processing" || j.state === "rendering");
  $("#job-progress-wrap").classList.toggle("hidden", !running);
  if (running) {
    const total = pr.total || 0;
    const pct = total ? Math.round(100 * (pr.frame || 0) / total) : 0;
    $("#job-fill").style.width = pct + "%";
    $("#pi-frames").textContent = "frame " + (pr.frame || 0) + " / " + (total || "?") + " (" + pct + "%)";
    $("#pi-fps").textContent = "speed: " + (pr.fps ? Number(pr.fps).toFixed(2) + " fps" : "-");
    $("#pi-eta").textContent = "remaining: " + fmtEta(pr.eta_s);
    $("#pi-stage").textContent = "stage: " + (STAGE_LABELS[pr.stage] || pr.stage || "-");
  }

  // error
  const errBox = $("#job-error");
  errBox.classList.toggle("hidden", j.state !== "error");
  if (j.state === "error") errBox.textContent = "Error: " + (j.error || "unknown");

  // stats + preview when processing finished (done OR already rendered)
  const finished = (j.state === "done" || j.state === "rendered");
  const stats = j.stats || null;
  const cards = $("#stats-cards");
  cards.classList.toggle("hidden", !(finished && stats));
  const warnBox = $("#job-warnings");
  warnBox.classList.add("hidden");
  if (finished && stats) {
    cards.innerHTML = "";
    const items = [
      ["Frames", stats.frames],
      ["Frames with conflict", stats.flagged_frames],
      ["M2 (conflicts)", (100 * (stats.m2_conflict_rate || 0)).toFixed(1) + "%"],
      ["Objects", stats.n_tracks],
      ["Conflicts", stats.n_conflicts],
      ["Pipeline FPS", stats.fps_pipeline],
      ["Peak VRAM", stats.peak_vram_gb !== null && stats.peak_vram_gb !== undefined ? stats.peak_vram_gb + " GB" : "-"],
    ];
    for (const it of items) {
      cards.append(el("div", { class: "stat-card" },
        el("div", { class: "v" }, String(it[1] === null || it[1] === undefined ? "-" : it[1])),
        el("div", { class: "k" }, it[0])));
    }
    if (stats.warnings && stats.warnings.length) {
      warnBox.classList.remove("hidden");
      warnBox.textContent = "Warnings: " + stats.warnings.join("; ");
    }
  }

  const vid = $("#preview-video");
  if (finished) {
    const url = mediaUrl("/api/jobs/" + j.job_id + "/preview.mp4");
    if (vid.dataset.src !== url) { vid.src = url; vid.dataset.src = url; }
    vid.classList.remove("hidden");
  } else {
    vid.classList.add("hidden");
  }

  const goto_ = $("#btn-goto-conflicts");
  goto_.classList.toggle("hidden", !finished);
  if (finished) {
    const n = stats ? stats.n_conflicts : "?";
    goto_.textContent = "Go to conflicts (" + n + ")";
  }
}

/* ---------- job list ---------- */

async function refreshJobs() {
  try {
    state.jobs = await api("/api/jobs");
  } catch (e) { return; }
  renderJobList();
  renderCfJobSelect();
}

function renderJobList() {
  const box = $("#job-list");
  box.innerHTML = "";
  if (!state.jobs.length) {
    box.append(el("div", { class: "muted" }, "No jobs."));
    return;
  }
  for (const j of state.jobs) {
    const bcls = j.state === "error" ? "err" : (j.state === "done" || j.state === "rendered") ? "ok" : "run";
    const sub = fmtDate(j.created_utc) +
      (j.stats && j.stats.n_conflicts !== undefined ? " | conflicts: " + j.stats.n_conflicts : "");
    box.append(el("div", { class: "job-row" },
      el("div", { class: "jr-main" },
        el("div", { class: "jr-name" }, (j.video_filename || j.video_id) + " (" + j.job_id + ")"),
        el("div", { class: "jr-sub" }, sub)),
      el("span", { class: "badge " + bcls }, STATE_LABELS[j.state] || j.state),
      el("button", { type: "button", onclick: () => openJob(j.job_id) }, "Open"),
    ));
  }
}

function bindJobControls() {
  $("#btn-process").addEventListener("click", startJob);
  $("#btn-refresh-jobs").addEventListener("click", refreshJobs);
  $("#btn-goto-conflicts").addEventListener("click", () => {
    switchTab("conflicts");
    if (state.jobId) openConflicts(state.jobId);
  });
}

/* =========================================================================
 * TAB 2 - Conflicts (the resolver)
 * ========================================================================= */

function normalizeDecisions(d) {
  d = d || {};
  return {
    tracks: d.tracks || {},
    frames: d.frames || {},
    dropped_frames: (d.dropped_frames || []).map(Number),
    unresolved_policy: d.unresolved_policy === "carve" ? "carve" : "failsafe",
  };
}

// SPEC section 3: resolved if frame dropped OR frame-local decision OR track
// decision; frame-local OVERRIDES track on that frame; drop wins for display.
function resolutionOf(c) {
  const d = state.cf.decisions;
  const f = String(c.frame), t = String(c.track_id);
  if (d.dropped_frames.indexOf(c.frame) !== -1) {
    return { resolved: true, by: "drop", decision: "drop_frame" };
  }
  if (d.frames[f] && d.frames[f][t]) {
    return { resolved: true, by: "frame", decision: d.frames[f][t] };
  }
  if (d.tracks[t]) {
    return { resolved: true, by: "track", decision: d.tracks[t] };
  }
  return { resolved: false, by: null, decision: null };
}

function badgeFor(res) {
  if (!res.resolved) return el("span", { class: "badge unres" }, "unresolved");
  if (res.by === "drop") return el("span", { class: "badge ok" }, "frame removed");
  const name = DECISION_SHORT[res.decision] || res.decision;
  if (res.by === "track") return el("span", { class: "badge ok" }, name + " (via object decision)");
  return el("span", { class: "badge ok" }, name);
}

/* ---------- decision mutations (toggle semantics) ---------- */

// returns true if a decision was APPLIED, false if it was CLEARED (toggle)
function applyDecision(c, key) {
  const d = state.cf.decisions;
  const f = String(c.frame), t = String(c.track_id);
  let applied = false;
  if (key === "unblur_track" || key === "blur_track") {
    if (d.tracks[t] === key) delete d.tracks[t];
    else { d.tracks[t] = key; applied = true; } // setting one clears the other
  } else if (key === "unblur_frame" || key === "blur_frame") {
    if (!d.frames[f]) d.frames[f] = {};
    if (d.frames[f][t] === key) {
      delete d.frames[f][t];
      if (!Object.keys(d.frames[f]).length) delete d.frames[f];
    } else { d.frames[f][t] = key; applied = true; }
  } else if (key === "drop_frame") {
    const i = d.dropped_frames.indexOf(c.frame);
    if (i !== -1) d.dropped_frames.splice(i, 1);
    else { d.dropped_frames.push(c.frame); d.dropped_frames.sort((a, b) => a - b); applied = true; }
  }
  scheduleSave();
  return applied;
}

function trackConflictCount(tid, onlyUnresolved) {
  let n = 0;
  for (const c of state.cf.data.conflicts) {
    if (c.track_id !== tid) continue;
    if (onlyUnresolved && resolutionOf(c).resolved) continue;
    n += 1;
  }
  return n;
}

function applyTrackDecision(tid, key) {
  const t = String(tid);
  const d = state.cf.decisions;
  const nConf = trackConflictCount(tid, false);
  let applied = false;
  if (d.tracks[t] === key) delete d.tracks[t];
  else { d.tracks[t] = key; applied = true; }
  scheduleSave();
  if (applied) toast("Resolved " + nConf + " conflicts of object #" + tid);
  else toast("Cleared decision for object #" + tid);
  renderConflictsTab();
  if (applied) maybeAutoAdvance();
}

/* ---------- auto-save (debounced 800 ms) ---------- */

function setSavedIndicator(mode) {
  const e = $("#cf-saved");
  if (mode === "dirty") { e.className = "saved-ind dirty"; e.textContent = "unsaved changes..."; }
  else if (mode === "saving") { e.className = "saved-ind dirty"; e.textContent = "saving..."; }
  else if (mode === "saved") { e.className = "saved-ind"; e.textContent = "saved"; }
  else if (mode === "error") { e.className = "saved-ind err"; e.textContent = "save error"; }
  else { e.className = "saved-ind"; e.textContent = ""; }
}

function scheduleSave() {
  state.cf.dirty = true;
  setSavedIndicator("dirty");
  clearTimeout(state.cf.saveTimer);
  state.cf.saveTimer = setTimeout(saveDecisionsNow, 800);
}

async function saveDecisionsNow() {
  if (!state.cf.jobId) return false;
  clearTimeout(state.cf.saveTimer);
  state.cf.saveTimer = null;
  setSavedIndicator("saving");
  try {
    await api("/api/jobs/" + state.cf.jobId + "/decisions", { method: "POST", body: state.cf.decisions });
    state.cf.dirty = false;
    setSavedIndicator("saved");
    return true;
  } catch (e) {
    setSavedIndicator("error");
    toast("Error saving decisions: " + e.message);
    return false;
  }
}

/* ---------- loading a job into the resolver ---------- */

async function openConflicts(jobId) {
  try {
    const results = await Promise.all([
      api("/api/jobs/" + jobId),
      api("/api/jobs/" + jobId + "/conflicts"),
      api("/api/jobs/" + jobId + "/decisions"),
    ]);
    state.cf.jobId = jobId;
    state.cf.job = results[0];
    state.cf.data = results[1];
    state.cf.decisions = normalizeDecisions(results[2]);
    state.cf.frame = state.cf.data.conflict_frames.length ? state.cf.data.conflict_frames[0] : null;
    state.cf.selIdx = 0;
    state.cf.dirty = false;
    setSavedIndicator("");
    localStorage.setItem(LS_LAST_JOB, jobId);
    stopRenderPoll();
    if (state.cf.job.state === "rendering") startRenderPoll();
    renderConflictsTab();
  } catch (e) {
    toast("Error loading conflicts: " + e.message);
  }
}

function renderCfJobSelect() {
  const sel = $("#cf-job-select");
  const cur = state.cf.jobId || "";
  sel.innerHTML = "";
  sel.append(el("option", { value: "" }, "- select -"));
  for (const j of state.jobs) {
    if (j.state !== "done" && j.state !== "rendering" && j.state !== "rendered") continue;
    sel.append(el("option", { value: j.job_id },
      (j.video_filename || j.video_id) + " (" + j.job_id + ", " + (STATE_LABELS[j.state] || j.state) + ")"));
  }
  sel.value = cur;
}

/* ---------- frame navigation ---------- */

function frameHasUnresolved(f) {
  return state.cf.data.conflicts.some((c) => c.frame === f && !resolutionOf(c).resolved);
}

function visibleFrames() {
  const all = state.cf.data.conflict_frames;
  if (!state.cf.onlyUnresolved) return all;
  const v = all.filter(frameHasUnresolved);
  return v.length ? v : all; // nothing left unresolved: show everything
}

function currentConflicts() {
  if (state.cf.frame === null || !state.cf.data) return [];
  return state.cf.data.conflicts.filter((c) => c.frame === state.cf.frame);
}

function gotoFrame(f) {
  state.cf.frame = f;
  state.cf.selIdx = 0;
  renderConflictsTab();
}

function navFrame(delta) {
  const vf = visibleFrames();
  if (!vf.length) return;
  let i = vf.indexOf(state.cf.frame);
  if (i === -1) {
    // current frame filtered out: snap to the nearest following frame
    i = vf.findIndex((f) => f >= state.cf.frame);
    if (i === -1) i = vf.length - 1;
    if (delta > 0) delta = 0;
  }
  const ni = Math.max(0, Math.min(vf.length - 1, i + delta));
  gotoFrame(vf[ni]);
}

function cycleCard(dir) {
  const cur = currentConflicts();
  if (!cur.length) return;
  state.cf.selIdx = ((state.cf.selIdx + dir) % cur.length + cur.length) % cur.length;
  renderConflictsTab();
}

/* ---------- applying options from cards / keyboard ---------- */

function selectedConflict() {
  const cur = currentConflicts();
  if (!cur.length) return null;
  const i = Math.max(0, Math.min(cur.length - 1, state.cf.selIdx));
  return cur[i];
}

function applyToSelected(key) {
  const c = selectedConflict();
  if (!c) return;
  const applied = applyDecision(c, key);
  renderConflictsTab();
  if (applied && resolutionOf(c).resolved) maybeAutoAdvance(c);
}

function maybeAutoAdvance(fromConflict) {
  if (!state.cf.autoAdvance || !state.cf.data) return;
  const list = state.cf.data.conflicts;
  if (!list.length) return;
  const from = fromConflict || selectedConflict();
  // Stay on the current frame until EVERY conflict on it is resolved; only
  // then move on. The reviewer decides each conflict of a frame in turn.
  const curFrame = (from && from.frame !== undefined && from.frame !== null)
    ? from.frame : state.cf.frame;
  if (curFrame !== null && curFrame !== undefined) {
    const onFrame = list.filter((x) => x.frame === curFrame);
    const fromIdx = from ? onFrame.indexOf(from) : -1;
    for (let s = 1; s <= onFrame.length; s++) {
      const c = onFrame[(fromIdx + s) % onFrame.length];
      if (!resolutionOf(c).resolved) {
        state.cf.frame = curFrame;
        state.cf.selIdx = onFrame.indexOf(c);
        renderConflictsTab();
        return;
      }
    }
  }
  // Current frame fully resolved: jump to the next unresolved conflict.
  let startIdx = from ? list.indexOf(from) : -1;
  if (startIdx === -1) startIdx = 0;
  for (let s = 1; s <= list.length; s++) {
    const i = (startIdx + s) % list.length;
    const c = list[i];
    if (!resolutionOf(c).resolved) {
      state.cf.frame = c.frame;
      const onFrame = list.filter((x) => x.frame === c.frame);
      state.cf.selIdx = Math.max(0, onFrame.indexOf(c));
      renderConflictsTab();
      return;
    }
  }
  // nothing unresolved left
  renderConflictsTab();
}

/* ---------- render functions for tab 2 ---------- */

function renderConflictsTab() {
  const has = !!state.cf.data;
  $("#cf-empty").classList.toggle("hidden", has);
  $("#cf-body").classList.toggle("hidden", !has);
  renderCfJobSelect();
  if (!has) {
    $("#cf-counts").textContent = "Conflicts: 0 | Resolved: 0 | Remaining: 0";
    return;
  }
  renderCfSummary();
  renderTrackList();
  renderViewer();
  renderCards();
  renderRenderPanel();
}

function renderCfSummary() {
  const all = state.cf.data.conflicts;
  const resolved = all.filter((c) => resolutionOf(c).resolved).length;
  $("#cf-counts").textContent =
    "Conflicts: " + all.length + " | Resolved: " + resolved + " | Remaining: " + (all.length - resolved);
  $("#cf-only-unresolved").checked = state.cf.onlyUnresolved;
  $("#cf-auto-advance").checked = state.cf.autoAdvance;
}

function renderTrackList() {
  const box = $("#track-list");
  box.innerHTML = "";
  const tracks = Object.values(state.cf.data.tracks || {})
    .slice().sort((a, b) => a.track_id - b.track_id);
  let shown = 0;
  for (const t of tracks) {
    const unres = trackConflictCount(t.track_id, true);
    if (state.cf.onlyUnresolved && unres === 0) continue;
    shown += 1;
    const tdec = state.cf.decisions.tracks[String(t.track_id)] || null;
    const row = el("div", { class: "track-row" },
      el("div", { class: "track-head" },
        el("span", { class: "cdot", style: "background:" + trackColor(t.track_id) }),
        el("b", {}, "#" + t.track_id + " " + (t.label || "object")),
        tdec ? el("span", { class: "badge ok" }, DECISION_SHORT[tdec] || tdec) : null),
      el("div", { class: "track-sub" },
        "frames " + t.first_frame + "-" + t.last_frame +
        " | conflicts: " + t.n_conflicts +
        " | unresolved: " + unres),
      el("div", { class: "track-btns" },
        el("button", {
          class: "opt-btn small" + (tdec === "unblur_track" ? " active" : ""),
          type: "button",
          onclick: () => applyTrackDecision(t.track_id, "unblur_track"),
        }, el("span", { class: "opt-num" }, "1"), " Don't blur everywhere"),
        el("button", {
          class: "opt-btn small" + (tdec === "blur_track" ? " active" : ""),
          type: "button",
          onclick: () => applyTrackDecision(t.track_id, "blur_track"),
        }, el("span", { class: "opt-num" }, "4"), " Blur everywhere")));
    box.append(row);
  }
  if (!shown) {
    box.append(el("div", { class: "muted" },
      state.cf.onlyUnresolved ? "No objects with unresolved conflicts." : "No objects with conflicts."));
  }
}

function renderViewer() {
  const img = $("#cf-frame-img");
  const label = $("#frame-label");
  const strip = $("#cf-strip");
  const vf = visibleFrames();
  $("#mode-raw").classList.toggle("active", state.cf.mode === "raw");
  $("#mode-red").classList.toggle("active", state.cf.mode === "redacted");

  if (!state.cf.data.conflict_frames.length || state.cf.frame === null) {
    img.removeAttribute("src");
    img.style.display = "none";
    label.textContent = "No frames with conflicts.";
    strip.innerHTML = "";
    return;
  }
  img.style.display = "block";
  const f = state.cf.frame;
  const url = mediaUrl("/api/jobs/" + state.cf.jobId + "/frame/" + f + ".jpg",
    { mode: state.cf.mode, overlay: 1, width: 1280 });
  if (img.dataset.src !== url) { img.src = url; img.dataset.src = url; }

  const fps = (state.cf.data.video && state.cf.data.video.fps) || 30;
  const pos = vf.indexOf(f);
  label.textContent = "frame " + f + " - " + (f / fps).toFixed(2) + " s - " +
    (pos >= 0 ? (pos + 1) : "-") + "/" + vf.length;

  strip.innerHTML = "";
  for (const sf of vf) {
    strip.append(el("button", {
      class: (sf === f ? "current " : "") + (frameHasUnresolved(sf) ? "unres" : ""),
      type: "button",
      title: "frame " + sf + " (" + (sf / fps).toFixed(2) + " s)",
      onclick: () => gotoFrame(sf),
    }, String(sf)));
  }
}

function renderCards() {
  const box = $("#cf-cards");
  box.innerHTML = "";
  const cur = currentConflicts();
  if (!cur.length) {
    box.append(el("div", { class: "muted" }, "No conflicts on this frame."));
    return;
  }
  if (state.cf.selIdx >= cur.length) state.cf.selIdx = cur.length - 1;
  cur.forEach((c, i) => {
    const res = resolutionOf(c);
    const t = state.cf.data.tracks[String(c.track_id)] || null;
    const d = state.cf.decisions;
    const fk = String(c.frame), tk = String(c.track_id);
    const activeOf = {
      unblur_track: d.tracks[tk] === "unblur_track",
      blur_track: d.tracks[tk] === "blur_track",
      unblur_frame: !!(d.frames[fk] && d.frames[fk][tk] === "unblur_frame"),
      blur_frame: !!(d.frames[fk] && d.frames[fk][tk] === "blur_frame"),
      drop_frame: d.dropped_frames.indexOf(c.frame) !== -1,
    };
    const opts = OPTION_KEYS.map((key, k) => el("button", {
      class: "opt-btn" + (activeOf[key] ? " active" : ""),
      type: "button",
      onclick: (ev) => {
        ev.stopPropagation();
        state.cf.selIdx = i;
        applyToSelected(key);
      },
    }, el("span", { class: "opt-num" }, String(k + 1)), " " + OPTION_LABELS[key]));

    const card = el("div", {
      class: "conflict-card" + (i === state.cf.selIdx ? " selected" : ""),
      style: "border-left-color:" + trackColor(c.track_id),
      onclick: () => { state.cf.selIdx = i; renderConflictsTab(); },
    },
      el("div", { class: "cc-head" },
        el("span", { class: "cdot", style: "background:" + trackColor(c.track_id) }),
        el("b", {}, "#" + c.track_id + " " + (c.label || "object")),
        badgeFor(res)),
      el("div", { class: "cc-info" },
        "near-zone overlap: " + Math.round(100 * (c.overlap || 0)) + "%" +
        (t ? " | object in frames " + t.first_frame + "-" + t.last_frame +
             " (conflicts: " + t.n_conflicts + ")" : "")),
      el("div", { class: "cc-opts" }, opts));
    box.append(card);
  });
}

/* ---------- render panel (final video) ---------- */

function renderRenderPanel() {
  const all = state.cf.data.conflicts;
  const unresolved = all.filter((c) => !resolutionOf(c).resolved).length;
  const policy = state.cf.decisions.unresolved_policy;
  $("#render-policy").value = policy;

  const warn = $("#render-warning");
  if (unresolved > 0) {
    warn.className = "warn";
    warn.textContent = (policy === "failsafe")
      ? "Unresolved conflicts (" + unresolved + ") will be BLURRED (failsafe)."
      : "Warning: unresolved conflicts (" + unresolved + ") will stay SHARP in the near zone (carve).";
  } else {
    warn.className = "ok";
    warn.textContent = "All conflicts resolved.";
  }

  const job = state.cf.job;
  const btn = $("#btn-render");
  const status = $("#render-status");
  const rstats = $("#render-stats");
  const vid = $("#final-video");
  const links = $("#render-links");

  const rendering = job && job.state === "rendering";
  btn.disabled = !job || rendering;
  status.textContent = rendering
    ? "Rendering..." + (job.progress && job.progress.total
        ? " frame " + (job.progress.frame || 0) + " / " + job.progress.total
        : "")
    : "";

  const isRendered = job && job.state === "rendered";
  vid.classList.toggle("hidden", !isRendered);
  links.classList.toggle("hidden", !isRendered);
  rstats.textContent = "";
  if (isRendered) {
    const url = mediaUrl("/api/jobs/" + state.cf.jobId + "/final.mp4");
    // renderSeq bumps on every observed rendering -> rendered transition, so
    // the player always reloads after a re-render (render_stats can repeat).
    const srcKey = url + "#r" + (state.cf.renderSeq || 0);
    if (vid.dataset.src !== srcKey) {
      vid.src = url + (url.indexOf("?") === -1 ? "?" : "&") + "t=" + Date.now();
      vid.dataset.src = srcKey;
    }
    $("#dl-final").href = url;
    $("#dl-log").href = mediaUrl("/api/jobs/" + state.cf.jobId + "/log");
    const rs = job.render_stats;
    if (rs) {
      rstats.textContent = "Wrote " + rs.frames_written + " frames, dropped " + rs.frames_dropped +
        ". Conflicts: " + rs.conflicts_total + " (resolved " + rs.resolved +
        ", unresolved " + rs.unresolved + ", policy: " + rs.unresolved_policy + ").";
    }
  }
}

async function startRender() {
  if (!state.cf.jobId) return;
  if (state.cf.dirty && !(await saveDecisionsNow())) {
    // decisions must hit disk first - never render with stale decisions
    toast("Render aborted - failed to save decisions");
    return;
  }
  try {
    await api("/api/jobs/" + state.cf.jobId + "/render", {
      method: "POST",
      body: { unresolved_policy: state.cf.decisions.unresolved_policy },
    });
    toast("Render started");
    if (state.cf.job) state.cf.job.state = "rendering";
    renderRenderPanel();
    startRenderPoll();
  } catch (e) {
    toast("Render error: " + e.message);
  }
}

function stopRenderPoll() {
  if (state.cf.renderTimer) { clearInterval(state.cf.renderTimer); state.cf.renderTimer = null; }
}

function startRenderPoll() {
  stopRenderPoll();
  state.cf.renderTimer = setInterval(pollRenderOnce, 1500);
  pollRenderOnce();
}

async function pollRenderOnce() {
  if (!state.cf.jobId) { stopRenderPoll(); return; }
  let j;
  try {
    j = await api("/api/jobs/" + state.cf.jobId);
  } catch (e) { return; }
  const prevState = state.cf.job && state.cf.job.state;
  state.cf.job = j;
  if (j.state === "rendered" && prevState === "rendering") {
    state.cf.renderSeq = (state.cf.renderSeq || 0) + 1;
  }
  renderRenderPanel();
  if (j.state === "rendered") {
    stopRenderPoll();
    toast("Final video ready");
    refreshJobs();
  } else if (j.state === "error") {
    stopRenderPoll();
    toast("Render error: " + (j.error || "unknown"));
  } else if (j.state !== "rendering") {
    stopRenderPoll();
  }
}

/* ---------- tab 2 bindings + keyboard ---------- */

function bindConflictUI() {
  $("#cf-job-select").addEventListener("change", (e) => {
    if (e.target.value) openConflicts(e.target.value);
  });
  $("#cf-only-unresolved").addEventListener("change", (e) => {
    state.cf.onlyUnresolved = e.target.checked;
    renderConflictsTab();
  });
  $("#cf-auto-advance").addEventListener("change", (e) => {
    state.cf.autoAdvance = e.target.checked;
  });
  $("#mode-raw").addEventListener("click", () => { state.cf.mode = "raw"; renderConflictsTab(); });
  $("#mode-red").addEventListener("click", () => { state.cf.mode = "redacted"; renderConflictsTab(); });
  $("#btn-prev-frame").addEventListener("click", () => navFrame(-1));
  $("#btn-next-frame").addEventListener("click", () => navFrame(1));
  $("#render-policy").addEventListener("change", (e) => {
    state.cf.decisions.unresolved_policy = e.target.value;
    scheduleSave();
    renderRenderPanel();
  });
  $("#btn-render").addEventListener("click", startRender);
}

function bindKeyboard() {
  document.addEventListener("keydown", (e) => {
    if (state.tab !== "conflicts" || !state.cf.data) return;
    const tag = (e.target && e.target.tagName || "").toLowerCase();
    if (tag === "input" || tag === "select" || tag === "textarea") return;
    if (e.ctrlKey || e.metaKey || e.altKey) return;
    if (e.key === "ArrowLeft") { e.preventDefault(); navFrame(-1); }
    else if (e.key === "ArrowRight") { e.preventDefault(); navFrame(1); }
    else if (e.key === "Tab") { e.preventDefault(); cycleCard(e.shiftKey ? -1 : 1); }
    else if (e.key >= "1" && e.key <= "5") {
      e.preventDefault();
      applyToSelected(OPTION_KEYS[parseInt(e.key, 10) - 1]);
    }
  });
}

/* ---------- top-level rerender (after settings change) ---------- */

function renderAll() {
  renderVideoMeta();
  renderJobStatus();
  renderJobList();
  renderConflictsTab();
}

/* ---------- init ---------- */

function init() {
  loadSettings();
  bindHeader();
  bindTabs();
  bindUpload();
  bindParamsForm();
  bindJobControls();
  bindConflictUI();
  bindKeyboard();

  pollHealth();
  setInterval(pollHealth, 10000);

  refreshVideos();
  refreshJobs().then(() => {
    const last = localStorage.getItem(LS_LAST_JOB);
    if (last && state.jobs.some((j) => j.job_id === last)) {
      state.jobId = last;
      startJobPoll();
    }
  });
}

document.addEventListener("DOMContentLoaded", init);
</script>
</body>
</html>


In [ ]:
# Preload the models (sam3.pt download + default predictor + UniDepth) so
# the first processing job in the browser does not pay the warm-up cost.
import os
os.environ["UCAP_AUTH_TOKEN"] = AUTH_TOKEN
import ucap_server
ucap_server.preload_models()
print("Modele zaladowane - serwer bedzie gotowy od pierwszego zadania.")

In [ ]:
# Start the FastAPI server (uvicorn) in a daemon thread, then poll the local
# health endpoint until it answers (max ~60 s).
import threading, time, json, urllib.request

SERVER_THREAD = threading.Thread(target=ucap_server.run,
                                 kwargs={"port": PORT}, daemon=True)
SERVER_THREAD.start()

health_url = "http://127.0.0.1:%d/api/health" % PORT
ok = False
for _ in range(60):
    try:
        with urllib.request.urlopen(health_url, timeout=2) as resp:
            payload = json.loads(resp.read().decode("utf-8"))
        ok = True
        break
    except Exception:
        time.sleep(1)

if ok:
    print("Serwer dziala lokalnie:", health_url)
    print(json.dumps(payload, indent=2, ensure_ascii=False))
else:
    print("BLAD: serwer nie odpowiedzial w 60 s - sprawdz komunikaty bledow "
          "w poprzednich komorkach.")

In [ ]:
# Public URL via a cloudflared quick tunnel (trycloudflare.com, no account).
# Re-running this cell kills the previous tunnel and opens a fresh one.
import os, re, stat, subprocess, time, urllib.request
from IPython.display import HTML, display

CF_BIN = "./cloudflared"
CF_URL = ("https://github.com/cloudflare/cloudflared/releases/latest/"
          "download/cloudflared-linux-amd64")

if not os.path.exists(CF_BIN):
    print("pobieranie cloudflared...")
    urllib.request.urlretrieve(CF_URL, CF_BIN)
    os.chmod(CF_BIN, os.stat(CF_BIN).st_mode | stat.S_IEXEC)

# Keep the process handle global so a re-run kills the old tunnel first.
if "CF_PROC" in globals() and CF_PROC is not None and CF_PROC.poll() is None:
    print("zamykanie poprzedniego tunelu...")
    CF_PROC.terminate()
    try:
        CF_PROC.wait(timeout=10)
    except Exception:
        CF_PROC.kill()

CF_PROC = subprocess.Popen(
    [CF_BIN, "tunnel", "--url", "http://localhost:%d" % PORT, "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# cloudflared logs the public URL on stderr - parse it (max ~60 s).
url = None
deadline = time.time() + 60
while time.time() < deadline and url is None:
    if CF_PROC.poll() is not None:
        print("BLAD: cloudflared zakonczyl sie przedwczesnie.")
        break
    line = CF_PROC.stderr.readline().decode("utf-8", errors="ignore")
    if not line:
        time.sleep(0.2)
        continue
    m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)

if url:
    print("=" * 70)
    print("  TWOJ PUBLICZNY ADRES UCAP TOOL:")
    print("  " + url)
    print("=" * 70)
    if AUTH_TOKEN:
        print("Token jest wlaczony - wpisz go w interfejsie po kliknieciu "
              "przycisku \"Settings\" w naglowku.")
    display(HTML(
        '<div style="margin:14px 0"><a href="' + url + '" target="_blank" '
        'style="font-size:30px;font-weight:bold;color:#4fc3f7;">' + url +
        '</a><br><span style="font-size:14px">Kliknij, aby otworzyc UCAP Tool '
        'w nowej karcie.</span></div>'))
else:
    print("Nie udalo sie odczytac adresu tunelu w 60 s - uruchom te komorke "
          "ponownie.")

## Jak korzystac z narzedzia

1. **Zakladka "Processing"**: wgraj wideo (przeciagnij plik lub wybierz z dysku), ustaw parametry (prompty SAM3, prog blisko w metrach, polityke carve/failsafe, styl redakcji) i kliknij **"Process"**. Obserwuj pasek postepu; po zakonczeniu zobaczysz podglad wideo i statystyki (m.in. wspolczynnik konfliktow M2).
2. **Zakladka "Conflicts"**: kazdy konflikt (obiekt prywatnosci w strefie blisko) rozwiazujesz jedna z 5 opcji: "Don't blur - whole object (all frames)" / "Don't blur - this frame only" / "Blur - this frame only" / "Blur - whole object (all frames)" / "Remove frame". Decyzje dla calego obiektu rozwiazuja od razu wszystkie jego konflikty (sledzenie SAM3). Decyzje zapisuja sie automatycznie.
3. Kliknij **"Render final video"** - nierozwiazane konflikty zostana domyslnie ZABLUROWANE (failsafe). Pobierz `final.mp4` i `decision_log.json` (pelny dziennik decyzji do audytu).

## Rozwiazywanie problemow

- **Blad 401 przy pobieraniu sam3.pt**: repozytorium `facebook/sam3` jest bramkowane - popros o dostep na https://huggingface.co/facebook/sam3, sprawdz sekret `HF_TOKEN` i uruchom notatnik ponownie.
- **Brak pamieci GPU (OOM)**: w `ucap_server.py` zmien model UniDepth na lzejszy (`lpiccinelli/unidepth-v2-vitb14` lub `...-vits14`) albo obniz `unidepth_res_level` w parametrach zadania; pomaga tez mniejsze `max_frames`.
- **Link przestal dzialac (tunel padl)**: uruchom ponownie OSTATNIA komorke z kodem - powstanie nowy adres trycloudflare.com.
- **Colab rozlaczyl sie z powodu bezczynnosci**: serwer znika razem ze srodowiskiem - zostaw karte Colaba otwarta podczas pracy.
- **Trwalosc danych**: wgrane wideo i wyniki zyja tylko tak dlugo, jak srodowisko wykonawcze. Pobierz `final.mp4` i `decision_log.json` zanim zamkniesz Colaba.
- **Prywatnosc linku**: adres trycloudflare.com jest publiczny, ale nieindeksowany. Jesli chcesz ochrony, ustaw `AUTH_TOKEN` w komorce CONFIG i uruchom notatnik ponownie.